# R2 Dialogue Log Verification System

    "This notebook runs **both** verification layers for Round 2 in one place:
- **Layer 1 (Steps 3a/3b)**: Pair comparison of Gemini-3.1 vs GPT-5.2, selection + targeted correction
- **Layer 2 (Step 3c)**: Second-pass recheck of corrected/custom rows to ensure fixes did not introduce new issues

Generated seeds from Gemini-3.1 and GPT-5.2 are evaluated using Claude Sonnet 4.5 as an independent arbiter with binary pass/fail validation on 5 criteria.\n",

## Execution Flow

The cells execute in 7 sequential steps:
1. Import libraries and set file paths
2. Load Gemini-3.1 and GPT-5.2 CSV datasets
3. Review binary validation system and decision logic
4. Initialize Claude Sonnet 4.5 verifier and define verification function
5. Run verification loop (currently limited to first 2 games for testing)
6. Create output DataFrames from verification results with per-criterion tracking
7. Save verified results to CSV files

## Output Files

1. **Datasets/verified/verified_r2_seeds_combined.csv**: Selected dialogues with source labels
2. **Datasets/verified/verified_r2_criteria_scores.csv**: Detailed per-criterion pass/fail tracking for analysis

## Step 1: Import Libraries

Import required Python libraries and set file paths.

In [14]:
# Import Required Libraries
import pandas as pd
import json
import os
import re
from typing import Dict, Tuple, Optional
from openai import OpenAI
import time
import random

# Initialize paths
DATA_DIR = "."
GEMINI3_FILE = "Datasets/seeds/generated_r2_seeds_gemini3.csv"
GPT52_FILE = "Datasets/seeds/generated_r2_seeds_gpt5_2.csv"
OUTPUT_FILE_COMBINED = "Datasets/verified/verified_r2_seeds_combined.csv"
OUTPUT_FILE_CRITERIA = "Datasets/verified/verified_r2_criteria_scores.csv"
SUMMARY_FILE = "Datasets/summarizer/summaries_r1.csv"  # Prior round (R1) summaries

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Load Datasets

Load the Gemini-3.1 and GPT-5.2 generated seed datasets.

In [15]:
# Load Generated Seeds from GPT Models
gemini3_df = pd.read_csv(GEMINI3_FILE)
gpt52_df = pd.read_csv(GPT52_FILE)

# Load prior round summaries for prior_summary_gold
_summary_df = pd.read_csv(SUMMARY_FILE)
prior_summary_lookup = dict(zip(_summary_df['game_id'], _summary_df['round_summary']))
print(f"Prior round summaries loaded: {len(prior_summary_lookup)} games")
if prior_summary_lookup:
    _sample_game = next(iter(prior_summary_lookup))
    print(f"  Sample (game_id={_sample_game}): {prior_summary_lookup[_sample_game][:120]}...")

print(f"Gemini-3.1 dataset loaded: {gemini3_df.shape[0]} rows, {gemini3_df.shape[1]} columns")
print(f"GPT-5.2 dataset loaded: {gpt52_df.shape[0]} rows, {gpt52_df.shape[1]} columns")
print(f"\nColumns in Gemini-3.1: {list(gemini3_df.columns)}")
print(f"Sample row (first game):")
print(f"  game_id: {gemini3_df.iloc[0]['game_id']}")
print(f"  player_roles: {gemini3_df.iloc[0]['player_roles'][:80]}...")
print(f"  matrix_tactic_scale sample: {gemini3_df.iloc[0]['matrix_tactic_scale'][:100]}...")

Prior round summaries loaded: 250 games
  Sample (game_id=G001): Round 1 Summary:
Leader: P4 | Team: P1, P4 | Vote: Unanimous | Outcome: FAIL

After the failed quest, P4 immediately bla...
Gemini-3.1 dataset loaded: 250 rows, 9 columns
GPT-5.2 dataset loaded: 250 rows, 9 columns

Columns in Gemini-3.1: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale']
Sample row (first game):
  game_id: G001
  player_roles: {"P1":"Good",
  "P2":"Good",
  "P3":"Evil",
  "P4":"Evil",
  "P5":"Good"}...
  matrix_tactic_scale sample: {
  "P5": {"row":"Selective / Framing","col":"Cooperative","tactic":"Bayesian hedging","scale":"GRS"...


## Step 3: Binary Multi-Criteria Validation System

**Validation Criteria:**

Claude Sonnet 4.5 evaluates each dialogue using **binary pass/fail checks** on 5 criteria:

1. **Discussion Coherence & Tactic Situational Fit** (1/0) - Each player's tactic makes situational sense given the social dynamics of the discussion
2. **Public History Alignment** (1/0) - Dialogue contextually appropriate to game state
3. **Tactic-Dialogue Alignment** (1/0) - Annotation accurately describes what was said, with correct taxonomy, scale, and alignment
4. **Authenticity & Skill Consistency** (1/0) - Natural dialogue with skill-modulated sophistication
5. **Dialogue Format Compliance** (1/0) - Starts with proper header, exactly 4 non-protagonist speakers, each speaks once

**Decision Logic:**

1. **Both 5/5 pass** → Claude Sonnet 4.5 performs pairwise comparison to break tie
2. **One 5/5, other <5/5** → Select the 5/5 response
3. **At least one 4/5**:
   - One 4/5, other <4/5 → Choose 4/5, **Targeted Correction** (fix only the failing criterion)
   - Both 4/5 → Claude's pairwise judgment (same holistic comparison as 5/5 tiebreaker), **Targeted Correction**
4. **Both ≤3/5** → **Full Custom Generation** (generate from scratch)

## Step 3a: Initialize Claude Sonnet 4.5

Set up the Anthropic client and system prompt.

In [16]:
# Initialize Claude Sonnet 4.5 Verifier
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI(
    api_key=os.getenv('ANTHROPIC_API_KEY'),
    base_url="https://api.anthropic.com/v1/",
)

SYSTEM_PROMPT = """You are an expert verifier for the Avalon social deception game with deep knowledge of deception tactics and game mechanics. Your task is to evaluate pairs of AI-generated dialogue responses, assess their quality, and determine the best outcome: select the superior response as-is, apply targeted corrections to fix specific issues in the better response, or generate an entirely new dialogue with appropriate tactics and scales if both responses are inadequate."""

# ── Build compact tactic taxonomy reference from knowledge base ───────────────
with open('tactics_knowledge_base.json', 'r') as _f:
    _kb = json.load(_f)

_COLOR_LABEL = {'green': 'Good-only', 'red': 'Evil-only', 'blue': 'Both alignments'}
_COLOR_ORDER = ['green', 'red', 'blue']
_grouped = {c: [] for c in _COLOR_ORDER}
for cell_key, cell in _kb['behavior_matrix'].items():
    row, col = cell_key.split('|')
    for tactic in cell['tactics']:
        _grouped[cell['color']].append((row, col, tactic))

_scale_lines = []
for scale_key, sd in _kb['scale_definitions'].items():
    level_strs = '; '.join(f'{lvl}: {desc}' for lvl, desc in sd['levels'].items())
    _scale_lines.append(f'  {scale_key} ({sd["name"]}) — for {sd["for"]}: {level_strs}')

_tax_lines = ['TACTIC ANNOTATION TAXONOMY']
_tax_lines.append('')
_tax_lines.append('The 4×4 Behavioral Matrix has two dimensions:')
_tax_lines.append('  Row (Information Strategy): Transparent | Selective/Framing | Careless-to-truth | Counterfactual')
_tax_lines.append('    - Transparent: speaker openly shares clear evidence-based information')
_tax_lines.append('    - Selective/Framing: speaker shapes communication through selective disclosure, emphasis, or framing')
_tax_lines.append('    - Careless-to-truth: speaker is indifferent to factual accuracy, i.e., bullshitting, vague speculation, or noise')
_tax_lines.append('    - Counterfactual: speaker asserts or believes information contrary to reality, i.e., honest mistake, self-deception, or hard lying')
_tax_lines.append('  Column (Goal Orientation): Cooperative | Defensive | Opportunistic | Adversarial')
_tax_lines.append('    - Cooperative: helps the group reach accurate conclusions')
_tax_lines.append('    - Defensive: protects self from suspicion or blame')
_tax_lines.append('    - Opportunistic: exploits the social moment for personal or team advantage')
_tax_lines.append('    - Adversarial: actively undermines specific players or misdirects the group')
_tax_lines.append('')
_tax_lines.append('Valid tactic names by alignment (use ONLY these exact names):')
for color in _COLOR_ORDER:
    _tax_lines.append(f'  {_COLOR_LABEL[color]}:')
    for row, col, tactic in _grouped[color]:
        _tax_lines.append(f'    [{row} | {col}] {tactic}')
_tax_lines.append('')
_tax_lines.append('Skill scales ("scale" field in matrix_tactic_scale):')
_tax_lines.extend(_scale_lines)
_tax_lines.append('  Note: Good players at Moderate/Low skill may legitimately use Blue (Both-alignment) tactics.')
_tax_lines.append('  Do NOT penalize a Good player for using Deflection, Self-deception, Face-saving blather,')
_tax_lines.append('  Vague filler, Noisy sincerity, Self-justification, Pragmatic silence, Omission to save face,')
_tax_lines.append('  Half-truths, or Rationalized error — these are valid defensive behaviors for Good players.')

TACTIC_TAXONOMY_REF = '\n'.join(_tax_lines)
print("\n" + TACTIC_TAXONOMY_REF + "\n")

print("✓ Claude Sonnet 4.5 system prompt initialized")
print(f"✓ Tactic taxonomy reference built: {len(_grouped['green']) + len(_grouped['red']) + len(_grouped['blue'])} tactics across 3 alignment groups")


TACTIC ANNOTATION TAXONOMY

The 4×4 Behavioral Matrix has two dimensions:
  Row (Information Strategy): Transparent | Selective/Framing | Careless-to-truth | Counterfactual
    - Transparent: speaker openly shares clear evidence-based information
    - Selective/Framing: speaker shapes communication through selective disclosure, emphasis, or framing
    - Careless-to-truth: speaker is indifferent to factual accuracy, i.e., bullshitting, vague speculation, or noise
    - Counterfactual: speaker asserts or believes information contrary to reality, i.e., honest mistake, self-deception, or hard lying
  Column (Goal Orientation): Cooperative | Defensive | Opportunistic | Adversarial
    - Cooperative: helps the group reach accurate conclusions
    - Defensive: protects self from suspicion or blame
    - Opportunistic: exploits the social moment for personal or team advantage
    - Adversarial: actively undermines specific players or misdirects the group

Valid tactic names by alignment (us

## Step 3b: Define Verification Function

Create the function that compares and evaluates dialogue pairs.

In [ ]:

# Debug flag — set to True to print the first prompt sent to Claude for each step
_PRINT_PROMPT_DEBUG = {'step3b': False, 'step3c': False}

def reorder_matrix_by_speaking_order(dialogue: str, matrix_input) -> str:
    """Reorder matrix_tactic_scale keys to match speaking order in the dialogue."""
    try:
        speakers = re.findall(r'(?m)^(P\d+):', dialogue or '')
        seen, ordered_speakers = set(), []
        for s in speakers:
            if s not in seen:
                seen.add(s)
                ordered_speakers.append(s)
        matrix = json.loads(matrix_input) if isinstance(matrix_input, str) else (matrix_input or {})
        reordered = {s: matrix[s] for s in ordered_speakers if s in matrix}
        for k, v in matrix.items():
            if k not in reordered:
                reordered[k] = v
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in reordered.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'
    except Exception:
        if isinstance(matrix_input, str):
            return matrix_input
        m = matrix_input or {}
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in m.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'


def verify_dialogue_pair(game_id: str,
                         round_id: str,
                         player_roles: str,
                         protagonist: str,
                         public_history: str,
                         dialogue_gemini3: str,
                         tactic_scale_gemini3: str,
                         dialogue_gpt52: str,
                         tactic_scale_gpt52: str,
                         prior_summary_gold: str = "") -> Dict:
    """
    Verify both dialogue responses using Claude Sonnet 4.5 with binary criteria validation.
    
    Returns: {
        'gemini3_criteria': {'coherence': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gemini3_total': int (0-5),
        'gpt52_criteria': {'coherence': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gpt52_total': int (0-5),
        'recommendation': str,
        'selected_dialogue': str,
        'selected_tactic_scale': str,
        'correction_mode': str (None, Targeted, Full_Custom, Pairwise_Tiebreaker),
        'failed_criterion': str or None,
        'reasoning': str,
        'needs_retry': bool  — True when tc_dialogue or cd_dialogue was None/empty (silent fallback applied)
    }
    """
    
    verification_prompt = f"""You are evaluating two AI-generated Avalon Round {round_id} discussion logs to determine which is superior. Both responses were generated for the same game context. Your task is to validate both using BINARY PASS/FAIL checks on 5 criteria, then recommend the best outcome: select one response as-is, apply a targeted correction to the better response, or generate a completely new dialogue with appropriate tactics and scales if both are inadequate.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Prior Round Summary:
{prior_summary_gold}

Public History (current round):
{public_history}

---

**RESPONSE A:**

Discussion Log:
{dialogue_gemini3}

Matrix Tactic Scale:
{tactic_scale_gemini3}

---

**RESPONSE B:**

Discussion Log:
{dialogue_gpt52}

Matrix Tactic Scale:
{tactic_scale_gpt52}

---

**TACTIC ANNOTATION FRAMEWORK:**
Both responses were generated from identical pre-assigned tactic plans. The Matrix Tactic Scale shown for each response is that model's annotation of its own dialogue. Neither is canonical. Treat them as starting references, not ground truth.

{TACTIC_TAXONOMY_REF}

Each speaker's annotation in matrix_tactic_scale must use this structure:
  "PX": {{"row": "<Information Strategy>", "col": "<Goal Orientation>", "tactic": "<exact name from taxonomy>", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Discussion Coherence & Tactic Situational Fit**:
   1 = Each player's tactic makes situational sense given the social dynamics of the discussion, that is, their conversational position, what other players are saying, and how tensions develop across the exchange
   0 = A player's tactic is socially incongruous given the discussion flow (e.g., "Pragmatic silence" for a player being directly accused by multiple others; "Vague filler" for a player everyone is turning to for information). The utterance may match the label literally, but the tactic choice is implausible for that social moment.

2. **Public History & Prior Round Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns from the current round's public history; also consistent with the Prior Round Summary context provided
   0 = Ignores prior events or contradicts established game state

3. **Tactic-Dialogue Alignment**:
   1 = Each player's annotation accurately describes what they actually said: the tactic name is a valid entry from the taxonomy, the row/col match the tactic's position in the 4×4 matrix, scale is GRS for Good players and Mach-IV for Evil players, and the level (High/Moderate/Low) is plausible given the sophistication of the utterance
   0 = The annotation mislabels what the player actually said (e.g., "Pragmatic silence" for a player who gave a 4-sentence analysis; "Hard lying" for a player who spoke honestly), OR a tactic name is not in the valid taxonomy, OR scale/row/col are inconsistent with the tactic definition

4. **Authenticity & Skill Consistency**:
   1 = The dialogue reads as a believable Avalon exchange with natural accusation/defense patterns, and the linguistic sophistication of each utterance is consistent with the speaker's assigned skill level.
   0 = The dialogue feels stilted or artificial, or the sophistication of an utterance clearly mismatches the assigned level (e.g., elaborate strategic reasoning from a Low-skill player, or uncertain hedging from a High-skill player).

5. **Dialogue Format Compliance**:
   1 = Starts with "Discussion after Quest X:", exactly 4 speakers (all non-protagonists), each speaks exactly once, in this format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong or missing header, incorrect speaker count, protagonist appears as speaker, or any format violations

**DECISION LOGIC:**

Apply this logic based on how many criteria each response passes:

1. **Both responses 5/5** → Use PAIRWISE_TIEBREAKER
   - Compare both dialogues to determine which is slightly better overall
   - Output RECOMMENDATION: PAIRWISE_TIEBREAKER

2. **One response 5/5, other <5/5** → Choose the 5/5 response
   - Output RECOMMENDATION: RESPONSE_A (if Response A is 5/5)
   - Output RECOMMENDATION: RESPONSE_B (if Response B is 5/5)

3. **At least one response 4/5** → Apply targeted correction
   - If only one is 4/5: choose that one → CORRECTION_A or CORRECTION_B
   - If both are 4/5: compare both holistically (same pairwise judgment as rule 1; which has stronger authenticity, history alignment, and tactic fit overall?) and choose the stronger base → CORRECTION_A or CORRECTION_B
   - Output RECOMMENDATION: CORRECTION_A or CORRECTION_B
   - The targeted_correction must correct the CHOSEN response (A or B)

4. **Both responses ≤3/5** → Generate custom dialogue
   - Output RECOMMENDATION: CUSTOM_GENERATION

**EVALUATION PROCESS:**

1. Evaluate RESPONSE A against all 5 criteria using binary PASS/FAIL checks
2. Evaluate RESPONSE B against all 5 criteria using the same binary checks
3. Count total passes for each response (X/5)
4. Apply the DECISION LOGIC rules based on the pass counts
5. Generate output in the exact format below

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object with the following structure:

```json
{{
  "response_a": {{
    "criteria": {{
      "coherence": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "response_b": {{
    "criteria": {{
      "coherence": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "recommendation": "RESPONSE_A / RESPONSE_B / PAIRWISE_TIEBREAKER / CORRECTION_A / CORRECTION_B / CUSTOM_GENERATION",
  "reasoning": "2-3 sentences explaining the decision",
  "pairwise_winner": "RESPONSE_A / RESPONSE_B / null",
  "pairwise_reasoning": "1-2 sentences explaining pairwise selection / null",
  "failed_criterion": "Coherence / History / Matrix / Authenticity / Format / null",
  "targeted_correction": {{
    "dialogue": "Full corrected dialogue text",
    "matrix_tactic_scale": {{"P2": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }},
  "custom_dialogue": {{
    "dialogue": "Complete new dialogue text",
    "matrix_tactic_scale": {{"P2": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }}
}}
```

**Field Explanations:**
- **score**: 1 = PASS, 0 = FAIL
- **explanation**: Brief explanation required for BOTH pass and fail to justify the score
- **total**: Integer count of passed criteria (0-5), NOT a fraction string
- **recommendation**: One of: "RESPONSE_A", "RESPONSE_B", "PAIRWISE_TIEBREAKER", "CORRECTION_A", "CORRECTION_B", "CUSTOM_GENERATION"
- **pairwise_winner**: Only if recommendation is PAIRWISE_TIEBREAKER (both 5/5), set to "RESPONSE_A" or "RESPONSE_B"; otherwise null
- **failed_criterion**: Only if recommendation is CORRECTION_A or CORRECTION_B, set to the failing criterion of the CHOSEN response: "Coherence", "History", "Matrix", "Authenticity", or "Format"; otherwise null
- **targeted_correction**: Corrects the CHOSEN response (matching recommendation — A or B). Only if failed_criterion is set; an object with:
  - "dialogue": full corrected dialogue text of the CHOSEN response, fixing its failed criterion
  - "matrix_tactic_scale": JSON mapping each speaker to tactic annotation; stay close to the tactic annotations shown in both matrices, updating only fields the corrected dialogue requires
  Otherwise null.
- **custom_dialogue**: Only if recommendation is CUSTOM_GENERATION (both ≤3/5); an object with:
  - "dialogue": complete new dialogue (exactly 4 speakers, protagonist excluded, "Discussion after Quest X:" format)
  - "matrix_tactic_scale": Use Response B's matrix_tactic_scale as the baseline for each speaker. Keep each speaker's tactic, row, col, scale, and level from Response B unless the new dialogue strictly requires a change. If any speaker's annotation must change, update only that speaker and ensure the revised tactic, row, col, scale, and level are valid under the tactic annotation framework and match what the new dialogue actually expresses. Generate entirely fresh dialogue content that authentically expresses those tactics in this specific game context.
  Otherwise null.
- Set **pairwise_winner** to null if recommendation is not PAIRWISE_TIEBREAKER
- Set **failed_criterion** to null if recommendation is not CORRECTION_A or CORRECTION_B
- Set **targeted_correction** to null if recommendation is not CORRECTION_A or CORRECTION_B
- Set **custom_dialogue** to null if recommendation is not CUSTOM_GENERATION (both ≤3/5)

**IMPORTANT:** Return ONLY the JSON object, no markdown code blocks, no extra text."""
    
    if _PRINT_PROMPT_DEBUG.get('step3b'):
        _PRINT_PROMPT_DEBUG['step3b'] = False
        print('=' * 80)
        print(f'[Step 3b] verification_prompt — {game_id}, Round {round_id}')
        print('=' * 80)
        print(verification_prompt)
        print('=' * 80)
    
    try:
        # Retry with exponential backoff for transient errors (rate limits, upstream failures)
        _MAX_RETRIES = 5
        _response_text = None
        for _retry in range(_MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model="claude-sonnet-4-5-20250929",
                    max_tokens=8192,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": verification_prompt}
                    ]
                )
                _response_text = response.choices[0].message.content
                break
            except Exception as _retry_e:
                if _retry < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _retry) + random.uniform(0, 2)
                    print(f"API error (attempt {_retry+1}/{_MAX_RETRIES}): {str(_retry_e)[:80]}. Retrying in {_delay:.1f}s...")
                    time.sleep(_delay)
                else:
                    raise
        response_text = _response_text
        
        # DEBUG: Print raw response to diagnose parsing issues
        # print(f"\n{'='*80}")
        # print(f"RAW CLAUDE RESPONSE FOR {game_id}:")
        # print(f"{'='*80}")
        # print(response_text)
        # print(f"{'='*80}\n")
        
        # Parse JSON response
        try:
            # Remove markdown code blocks if present
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                # Remove opening ``` line
                lines = lines[1:]
                # Find and remove closing ```
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)
            
            # Parse JSON
            data = json.loads(clean_text)
            
            # Build result dictionary from JSON
            result = {
                'gemini3_criteria': {
                    'coherence': data['response_a']['criteria']['coherence']['score'] == 1,
                    'history': data['response_a']['criteria']['history']['score'] == 1,
                    'matrix': data['response_a']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_a']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_a']['criteria']['format']['score'] == 1
                },
                'gemini3_total': int(data['response_a']['total']) if isinstance(data['response_a']['total'], int) else int(str(data['response_a']['total']).split('/')[0]),
                'gemini3_criteria_explanations': {
                    criterion: data['response_a']['criteria'][criterion].get('explanation', '')
                    for criterion in ['coherence', 'history', 'matrix', 'authenticity', 'format']
                },
                'gpt52_criteria': {
                    'coherence': data['response_b']['criteria']['coherence']['score'] == 1,
                    'history': data['response_b']['criteria']['history']['score'] == 1,
                    'matrix': data['response_b']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_b']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_b']['criteria']['format']['score'] == 1
                },
                'gpt52_total': int(data['response_b']['total']) if isinstance(data['response_b']['total'], int) else int(str(data['response_b']['total']).split('/')[0]),
                'gpt52_criteria_explanations': {
                    criterion: data['response_b']['criteria'][criterion].get('explanation', '')
                    for criterion in ['coherence', 'history', 'matrix', 'authenticity', 'format']
                },
                'recommendation': data['recommendation'],
                'reasoning': data['reasoning'],
                'pairwise_winner': data.get('pairwise_winner'),
                'pairwise_reasoning': data.get('pairwise_reasoning', ''),
                'failed_criterion': data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'custom_dialogue': data.get('custom_dialogue'),
                'selected_dialogue': None,
                'selected_tactic_scale': None,
                'correction_mode': None,
                'needs_retry': False,   # set True below when tc/cd silently falls back
            }
            
            # Determine final selection based on scores
            s1 = result['gemini3_total']
            s2 = result['gpt52_total']
            
            if s1 == 5 and s2 == 5:
                # Both 5/5 - use pairwise tiebreaker
                if result['pairwise_winner'] and 'RESPONSE_B' in result['pairwise_winner'].upper():
                    result['recommendation'] = 'RESPONSE_B'
                    result['selected_dialogue'] = dialogue_gpt52
                    result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gpt52, tactic_scale_gpt52)
                else:
                    result['recommendation'] = 'RESPONSE_A'
                    result['selected_dialogue'] = dialogue_gemini3
                    result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gemini3, tactic_scale_gemini3)
                result['correction_mode'] = 'Pairwise_Tiebreaker'
                
            elif s1 == 5:
                # Only Response A is 5/5
                result['recommendation'] = 'RESPONSE_A'
                result['selected_dialogue'] = dialogue_gemini3
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gemini3, tactic_scale_gemini3)
                result['correction_mode'] = None
                
            elif s2 == 5:
                # Only Response B is 5/5
                result['recommendation'] = 'RESPONSE_B'
                result['selected_dialogue'] = dialogue_gpt52
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gpt52, tactic_scale_gpt52)
                result['correction_mode'] = None
                
            elif s1 == 4 or s2 == 4:
                # At least one 4/5 - targeted correction
                tc          = result.get('targeted_correction') or {}
                tc_dialogue = tc.get('dialogue') if isinstance(tc, dict) else None
                tc_matrix   = tc.get('matrix_tactic_scale') if isinstance(tc, dict) else None
                
                if s1 == 4 and s2 == 4:
                    # Both 4/4 — trust Claude's pairwise judgment (same logic as 5/5 tiebreaker)
                    claude_rec = result.get('recommendation', 'CORRECTION_A')
                    if 'CORRECTION_A' == claude_rec:
                        result['recommendation'] = 'CORRECTION_A'
                        final_dialogue = tc_dialogue or dialogue_gemini3
                        final_matrix   = tc_matrix   or tactic_scale_gemini3
                    else:
                        result['recommendation'] = 'CORRECTION_B'
                        final_dialogue = tc_dialogue or dialogue_gpt52
                        final_matrix   = tc_matrix   or tactic_scale_gpt52
                elif s1 == 4:
                    result['recommendation'] = 'CORRECTION_A'
                    final_dialogue = tc_dialogue or dialogue_gemini3
                    final_matrix   = tc_matrix   or tactic_scale_gemini3
                else:  # s2 == 4
                    result['recommendation'] = 'CORRECTION_B'
                    final_dialogue = tc_dialogue or dialogue_gpt52
                    final_matrix   = tc_matrix   or tactic_scale_gpt52
                
                if tc_dialogue is None:
                    print(f"  ⚠️  WARNING: targeted_correction.dialogue is None for {game_id} "
                          f"(rec={result['recommendation']}) — falling back to uncorrected base dialogue")
                    result['needs_retry'] = True
                
                result['selected_dialogue']     = final_dialogue
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    final_dialogue,
                    json.dumps(final_matrix, separators=(',', ':')) if isinstance(final_matrix, dict) else final_matrix
                )
                result['correction_mode'] = 'Targeted'
                
            else:
                # Both ≤3/5 - full custom generation; anchor matrix to Response B's assignments
                cd          = result.get('custom_dialogue') or {}
                cd_dialogue = cd.get('dialogue') if isinstance(cd, dict) else None
                cd_matrix   = cd.get('matrix_tactic_scale') if isinstance(cd, dict) else None
                if not cd_dialogue:
                    print(f"  ⚠️  WARNING: Custom dialogue is empty for {game_id}!")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Full_Custom'
                result['selected_dialogue']     = cd_dialogue
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    cd_dialogue or '',
                    json.dumps(cd_matrix, separators=(',', ':')) if isinstance(cd_matrix, dict) else (cd_matrix or tactic_scale_gpt52)
                )
                result['recommendation'] = 'CUSTOM_GENERATION'
            return result, response_text
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            print(f"Cleaned text: {clean_text[:500]}...")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing JSON for {game_id}: {e}")
            return None, str(e)
    except Exception as e:
        print(f"Error verifying game {game_id}: {str(e)}")
        return None, str(e)


## Step 3c: Define verify_single_dialogue (Layer 2 Inline Verifier)

Defines `verify_single_dialogue()` — the single-response scorer used for inline Layer 2 rechecking.  
Also initialises `_CRITERIA_ORDER` and `_LABEL_MAP` used by Steps 4a and 4b.

**Why this cell must run before Step 4a:**  
The main verification loop (Step 4a) calls `verify_single_dialogue()` immediately after every Targeted correction or Full_Custom generation — up to 3 inline attempts.

In [ ]:

# ─── Step 3c: Shared constants + verify_single_dialogue() ────────────────────
# Must run BEFORE Step 4a so the inline Layer 2 recheck can call this function.
# ─────────────────────────────────────────────────────────────────────────────

_CRITERIA_ORDER = ['coherence', 'history', 'matrix', 'authenticity', 'format']
_LABEL_MAP = {
    'RESPONSE_A':        'Gemini-3.1',
    'CORRECTION_A':      'Gemini-3.1-Claude-4.5',
    'RESPONSE_B':        'GPT-5.2',
    'CORRECTION_B':      'GPT-5.2-Claude-4.5',
    'CUSTOM_GENERATION': 'Claude-4.5',
}


def verify_single_dialogue(game_id, round_id, player_roles, protagonist,
                            public_history, discussion_log, matrix_tactic_scale,
                            prior_summary_gold=""):
    """
    Evaluate ONE dialogue on the same 5 binary criteria as verify_dialogue_pair().
    Called inline within the Step 4a main loop immediately after every Targeted
    correction or Full_Custom generation — up to 3 attempts before NEEDS_HUMAN.

    Returns: (result_dict, raw_response)
    result_dict keys:
      criteria              — {coherence, history, matrix, authenticity, format} -> bool
      criteria_explanations — {criterion -> explanation str}
      total                 — int 0-5
      recommendation        — 'ACCEPT' | 'TARGETED_CORRECTION' | 'FULL_REGENERATION'
      failed_criterion      — str or None
      reasoning             — str
      selected_dialogue     — str (unchanged / corrected / regenerated)
      selected_tactic_scale — str (JSON)
      correction_mode       — 'Accepted' | 'Recorrected' | 'Regenerated'
      needs_retry           — bool
    """
    recheck_prompt = f"""You are evaluating a single AI-generated Avalon discussion log in Round {round_id} for quality. The dialogue has already undergone one round of correction.
Your default is to ACCEPT it; only flag a criterion as failing when there is a clear, objective, non-debatable violation. You must not be nitpicky or subjective. The goal is to ensure basic quality and consistency, not to find minor flaws.     
Score it on 5 binary criteria and, if it fails any criterion, you must apply the minimum fix required to pass that criterion, then re-evaluate the corrected dialogue against all criteria again.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Prior Round Summary:
{prior_summary_gold}

Public History (current round):
{public_history}

---

**DIALOGUE TO EVALUATE:**

Discussion Log:
{discussion_log}

Matrix Tactic Scale:
{matrix_tactic_scale}

---

**TACTIC ANNOTATION FRAMEWORK:**
{TACTIC_TAXONOMY_REF}

Each speaker's annotation in matrix_tactic_scale must use this structure:
  "PX": {{"row": "<Information Strategy>", "col": "<Goal Orientation>", "tactic": "<exact name from taxonomy>", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Discussion Coherence & Tactic Situational Fit**:
   1 = Each player's tactic makes situational sense given the social dynamics of the discussion, that is, their conversational position, what other players are saying, and how tensions develop across the exchange
   0 = A player's tactic is socially incongruous given the discussion flow (e.g., "Pragmatic silence" for a player being directly accused by multiple others; "Vague filler" for a player everyone is turning to for information). The utterance may match the label literally, but the tactic choice is implausible for that social moment.

2. **Public History & Prior Round Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns from the current round's public history; also consistent with the Prior Round Summary context provided
   0 = Ignores prior events or contradicts established game state

3. **Tactic-Dialogue Alignment**:
   1 = Each player's annotation accurately describes what they actually said: the tactic name is a valid entry from the taxonomy, the row/col match the tactic's position in the 4x4 matrix, scale is GRS for Good players and Mach-IV for Evil players, and the level (High/Moderate/Low) is plausible given the sophistication of the utterance
   0 = The annotation mislabels what the player actually said (e.g., "Pragmatic silence" for a player who gave a 4-sentence analysis; "Hard lying" for a player who spoke honestly), OR a tactic name is not in the valid taxonomy, OR scale/row/col are inconsistent with the tactic definition

4. **Authenticity & Skill Consistency**:
   1 = The dialogue reads as a believable Avalon exchange with natural accusation/defense patterns, and the linguistic sophistication of each utterance is consistent with the speaker's assigned skill level.
   0 = The dialogue feels stilted or artificial, or the sophistication of an utterance clearly mismatches the assigned level (e.g., elaborate strategic reasoning from a Low-skill player, or uncertain hedging from a High-skill player).

5. **Dialogue Format Compliance**:
   1 = Starts with "Discussion after Quest X:", exactly 4 speakers (all non-protagonists), each speaks exactly once, in this format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong or missing header, incorrect speaker count, protagonist appears as speaker, or any format violations

---

**DECISION LOGIC:**

1. **5/5** -> ACCEPT the dialogue as-is
2. **4/5** -> TARGETED_CORRECTION: fix the single failing criterion in the provided dialogue
3. **<=3/5** -> FULL_REGENERATION: generate a completely new dialogue that passes all 5 criteria

---

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object from one of the three decision paths: 1) ACCEPT, 2) TARGETED_CORRECTION, or 3) FULL_REGENERATION.

1. If the output is a full 5/5 ACCEPT:

```json
{{
  "criteria": {{
    "coherence": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "history": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "matrix": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "authenticity": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "format": {{"score": 1, "explanation": "Brief explanation why it passed"}}
  }},
  "total": 5,
  "recommendation": "ACCEPT",
  "reasoning": "2-3 sentences on overall quality",
  "failed_criterion": null,
  "targeted_correction": null,
  "regenerated_dialogue": null
}}
```

2. If exactly one criterion has a clear, objective failure (4/5), use TARGETED_CORRECTION:

```json
{{
  "criteria": {{
    "coherence": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "history": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "matrix": {{"score": 0, "explanation": "Brief explanation of the objective failure"}},
    "authenticity": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "format": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}}
  }},
  "total": 4,
  "recommendation": "TARGETED_CORRECTION",
  "reasoning": "2-3 sentences on overall quality and decision",
  "failed_criterion": "matrix",
  "targeted_correction": {{
    "dialogue": "Full corrected dialogue text fixing only the failed criterion",
    "matrix_tactic_scale": {{"P#": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "P#": {{...}}, "P#": {{...}}, "P#": {{...}}}}
  }},
  "regenerated_dialogue": null
}}
```

3. When recommendation is FULL_REGENERATION, regenerated_dialogue takes this structure (targeted_correction will be null):

```json
{{
  "criteria": {{
    "coherence": {{"score": 0, "explanation": "..."}},
    "history": {{"score": 0, "explanation": "..."}},
    "matrix": {{"score": 1, "explanation": "..."}},
    "authenticity": {{"score": 1, "explanation": "..."}},
    "format": {{"score": 1, "explanation": "..."}}
  }},
  "total": 3,
  "recommendation": "FULL_REGENERATION",
  "reasoning": "The dialogue coherence and history are critically broken; a complete rewrite is needed.",
  "failed_criterion": null,
  "targeted_correction": null,
  "regenerated_dialogue": {{
    "dialogue": "Discussion after Quest X:\\nPlayerA: [utterance]\\nPlayerB: [utterance]\\nPlayerC: [utterance]\\nPlayerD: [utterance]",
    "matrix_tactic_scale": {{"PX": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }}
}}
```

**Field rules:**
- **targeted_correction**: Populated only when recommendation is TARGETED_CORRECTION; an object with:
  - "dialogue": full corrected dialogue text fixing only the failed criterion
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; keep all speakers' entries from the input unless the correction strictly requires a change
  Otherwise null.
- **regenerated_dialogue**: Populated only when recommendation is FULL_REGENERATION; an object with:
  - "dialogue": complete new dialogue (exactly 4 speakers, protagonist excluded, "Discussion after Quest X:" header). Use actual newlines in the JSON string (newline character, not escaped \\n).
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; use the input matrix_tactic_scale as baseline — keep each speaker's row, col, tactic, scale, and level unless the new dialogue strictly requires a change
  Otherwise null.
- **failed_criterion**: The single failing criterion name (coherence/history/matrix/authenticity/format) when recommendation is TARGETED_CORRECTION; null otherwise
- **Benefit of the doubt rule**: When uncertain whether a flaw exists or whether it meets the 0/1 threshold, always score as passing (1). Only assign 0 when the failure is absolutely clear, objective, and non-debatable.
- Return ONLY the JSON object, no markdown code blocks, no extra text."""

    if _PRINT_PROMPT_DEBUG.get('step3c'):
        _PRINT_PROMPT_DEBUG['step3c'] = False
        print('=' * 80)
        print(f'[Step 3c] recheck_prompt — {game_id}, Round {round_id}')
        print('=' * 80)
        print(recheck_prompt)
        print('=' * 80)

    try:
        # Retry with exponential backoff for transient errors (rate limits, upstream failures)
        _MAX_RETRIES = 5
        _response_text = None
        for _retry in range(_MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model="claude-sonnet-4-5-20250929",
                    max_tokens=4096,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": recheck_prompt}
                    ]
                )
                _response_text = response.choices[0].message.content
                break
            except Exception as _retry_e:
                if _retry < _MAX_RETRIES - 1:
                    _delay = 5 * (2 ** _retry) + random.uniform(0, 2)
                    print(f"API error (attempt {_retry+1}/{_MAX_RETRIES}): {str(_retry_e)[:80]}. Retrying in {_delay:.1f}s...")
                    time.sleep(_delay)
                else:
                    raise
        response_text = _response_text

        try:
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                lines = lines[1:]
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)

            data     = json.loads(clean_text)
            criteria = data['criteria']

            result = {
                'criteria': {k: criteria[k]['score'] == 1 for k in _CRITERIA_ORDER},
                'criteria_explanations': {k: criteria[k].get('explanation', '') for k in _CRITERIA_ORDER},
                'total': int(data['total']) if isinstance(data['total'], int) else int(str(data['total']).split('/')[0]),
                'recommendation':       data.get('recommendation', 'ACCEPT'),
                'reasoning':            data.get('reasoning', ''),
                'failed_criterion':     data.get('failed_criterion'),
                'targeted_correction':  data.get('targeted_correction'),
                'regenerated_dialogue': data.get('regenerated_dialogue'),
                'selected_dialogue':     None,
                'selected_tactic_scale': None,
                'correction_mode':       None,
                'needs_retry':           False,
            }

            rec = result['recommendation']

            if result['total'] == 5 or rec == 'ACCEPT':
                result['correction_mode']       = 'Accepted'
                result['selected_dialogue']     = discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(discussion_log, matrix_tactic_scale)

            elif rec == 'TARGETED_CORRECTION':
                tc          = result.get('targeted_correction') or {}
                tc_dialogue = tc.get('dialogue') if isinstance(tc, dict) else None
                tc_matrix   = tc.get('matrix_tactic_scale') if isinstance(tc, dict) else None
                if tc_dialogue is None:
                    print(f"  ⚠️  WARNING: targeted_correction.dialogue is None for {game_id} — flagging for retry")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Recorrected'
                result['selected_dialogue']     = tc_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    tc_dialogue or discussion_log,
                    json.dumps(tc_matrix, separators=(',', ':')) if isinstance(tc_matrix, dict) else (tc_matrix or matrix_tactic_scale)
                )

            else:  # FULL_REGENERATION
                rd          = result.get('regenerated_dialogue') or {}
                rd_dialogue = rd.get('dialogue') if isinstance(rd, dict) else None
                rd_matrix   = rd.get('matrix_tactic_scale') if isinstance(rd, dict) else None
                if not rd_dialogue:
                    print(f"  ⚠️  WARNING: regenerated_dialogue is empty for {game_id}!")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Regenerated'
                result['selected_dialogue']     = rd_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    rd_dialogue or discussion_log,
                    json.dumps(rd_matrix, separators=(',', ':')) if isinstance(rd_matrix, dict) else (rd_matrix or matrix_tactic_scale)
                )

            return result, response_text

        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing recheck JSON for {game_id}: {e}")
            return None, str(e)

    except Exception as e:
        print(f"  Error rechecking game {game_id}: {str(e)}")
        return None, str(e)


print("✓ verify_single_dialogue() defined")
print("✓ _CRITERIA_ORDER and _LABEL_MAP initialized")


✓ verify_single_dialogue() defined
✓ _CRITERIA_ORDER and _LABEL_MAP initialized


## Step 4a: Run Verification Loop

Process games and collect verification results.

In [19]:
# Helper function to format matrix tactic scale with pretty-printing
def format_matrix_tactic_scale(json_str: str) -> str:
    """Format JSON with linebreaks for readability"""
    try:
        data = json.loads(json_str)
        # Reconstruct with pretty-printing: 2-space indent
        return json.dumps(data, indent=2)
    except:
        return json_str

def format_check(val):
    return 1 if val else 0

# Compare and Validate Dialogue Rows
# Initialize results storage
verified_results = []
criteria_results = []
verification_stats = {
    'total_games': 0,
    'response_1_selected': 0,
    'response_2_selected': 0,
    'targeted_correction_1': 0,
    'targeted_correction_2': 0,
    'pairwise_tiebreaker': 0,
    'full_custom_generation': 0,
    'needs_human': 0,     # rows flagged after 3 failed inline Layer 2 rechecks
    'errors': 0
}

# Tracks indices that need a retry in Step 4b.
# Each entry: {'idx': int, 'game_id': str, 'round_id': str/int, 'reason': str}
#   reason = 'api_error'   — verify_dialogue_pair returned None (API / parse failure)
#   reason = 'Targeted'    — tc_dialogue was None; fell back to uncorrected base
#   reason = 'Full_Custom' — cd_dialogue was empty; selected_dialogue stored as None
retry_rows = []

# Process all games. Change TEST_LIMIT to a small number (e.g. 2) for test runs or len(gemini3_df) for full runs.
# TEST_LIMIT = 1
TEST_LIMIT = len(gemini3_df)
print(f"Starting verification of {min(TEST_LIMIT, len(gemini3_df))} games...")
print("=" * 80)

for idx in range(min(TEST_LIMIT, len(gemini3_df))):
    game_id = gemini3_df.iloc[idx]['game_id']
    round_id = gemini3_df.iloc[idx]['round_id']
    score_id = f"{game_id}/{round_id}"

    # Extract fields from both datasets
    player_roles = gemini3_df.iloc[idx]['player_roles']
    protagonist  = gemini3_df.iloc[idx]['role_id']   # Protagonist who doesn't speak
    public_history = gemini3_df.iloc[idx]['public_history']
    prior_summary_gold = prior_summary_lookup.get(gemini3_df.iloc[idx]['game_id'], '')

    dialogue_gemini3    = gemini3_df.iloc[idx]['discussion_log']
    tactic_scale_gemini3 = gemini3_df.iloc[idx]['matrix_tactic_scale']

    dialogue_gpt52    = gpt52_df.iloc[idx]['discussion_log']
    tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']

    # ── Layer 1: Pair comparison ───────────────────────────────────────────────
    print(f"\n[{idx+1}/{min(TEST_LIMIT, len(gemini3_df))}] Verifying game {score_id}...")
    result, raw_response = verify_dialogue_pair(
        game_id, round_id, player_roles, protagonist, public_history,
        dialogue_gemini3, tactic_scale_gemini3,
        dialogue_gpt52, tactic_scale_gpt52,
        prior_summary_gold=prior_summary_gold,
    )

    if result is None:
        print(f"  ✗ Error: {raw_response}")
        verification_stats['errors'] += 1
        retry_rows.append({'idx': idx, 'game_id': game_id, 'round_id': round_id, 'reason': 'api_error'})
        continue

    verification_stats['total_games'] += 1

    # Determine LLM source and label for output (map Response A/B back to actual model names)
    if result['recommendation'] == 'RESPONSE_A':
        llm_used = 'Gemini-3.1'
        selected_row = gemini3_df.iloc[idx].copy()
        verification_stats['response_1_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_A':
        llm_used = 'Gemini-3.1-Claude-4.5'
        selected_row = gemini3_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['targeted_correction_1'] += 1
    elif result['recommendation'] == 'RESPONSE_B':
        llm_used = 'GPT-5.2'
        selected_row = gpt52_df.iloc[idx].copy()
        verification_stats['response_2_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_B':
        llm_used = 'GPT-5.2-Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['targeted_correction_2'] += 1
    else:  # CUSTOM_GENERATION
        llm_used = 'Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['full_custom_generation'] += 1

    # Use the already-ordered tactic scale from verify_dialogue_pair (fallback to GPT-5.2)
    selected_tactic_scale = result['selected_tactic_scale'] or format_matrix_tactic_scale(tactic_scale_gpt52)

    # Track pairwise tiebreakers separately
    if result['correction_mode'] == 'Pairwise_Tiebreaker':
        verification_stats['pairwise_tiebreaker'] += 1

    # ── Layer 2: Inline recheck for Targeted / Full_Custom corrections ─────────
    # After every correction, verify the corrected dialogue on the same 5 criteria
    # with up to 3 attempts. On 3 consecutive failures → flag row as NEEDS_HUMAN.
    _inl_mode   = 'N/A'
    _inl_score  = 'N/A'
    _inl_reason = 'N/A'
    _inl_expls  = 'N/A'

    if (
        result.get('correction_mode') in ('Targeted', 'Full_Custom')
        and not result.get('needs_retry')        # skip if Layer 1 already fell back (tc was None)
        and result.get('selected_dialogue')
    ):
        _inl_dialogue = result['selected_dialogue']
        _inl_matrix   = selected_tactic_scale

        for _att in range(1, 4):  # attempts 1, 2, 3
            time.sleep(1)
            print(f"  🔍 Inline L2 recheck attempt {_att}/3...")
            _vsd, _ = verify_single_dialogue(
                game_id, round_id, player_roles, protagonist, public_history,
                _inl_dialogue, _inl_matrix,
                prior_summary_gold=prior_summary_gold
            )

            if _vsd is None:
                print(f"  ⚠️  Inline recheck API error (attempt {_att})")
                if _att == 3:
                    print(f"  ⚠️  All 3 inline attempts failed — flagging NEEDS_HUMAN")
                    llm_used  += '-NEEDS_HUMAN'
                    _inl_mode  = 'NEEDS_HUMAN'
                    _inl_score  = '0 of 5'
                    _inl_reason = 'API error on all 3 inline attempts'
                    _inl_expls  = '{}'
                    verification_stats['needs_human'] += 1
                continue

            _inl_score  = f"{_vsd['total']} of 5"
            _inl_reason = _vsd['reasoning']
            _inl_expls  = json.dumps(_vsd['criteria_explanations'])

            if _vsd['total'] == 5:
                print(f"  ✓ Inline L2 recheck: 5/5 — accepted (attempt {_att})")
                _inl_dialogue = _vsd['selected_dialogue']
                _inl_matrix   = _vsd['selected_tactic_scale']
                _inl_mode     = 'INLINE_ACCEPTED'
                break
            elif _att < 3:
                print(f"  ⚠️  Inline L2: {_vsd['total']}/5 — fix applied, retrying (attempt {_att})")
                _inl_dialogue = _vsd['selected_dialogue']
                _inl_matrix   = _vsd['selected_tactic_scale']
                _inl_mode     = 'INLINE_FIXED'
            else:  # _att == 3 and total < 5
                print(f"  ⚠️  Inline L2: {_vsd['total']}/5 after 3 attempts — flagging NEEDS_HUMAN")
                _inl_dialogue = _vsd['selected_dialogue']
                _inl_matrix   = _vsd['selected_tactic_scale']
                llm_used  += '-NEEDS_HUMAN'
                _inl_mode  = 'NEEDS_HUMAN'
                verification_stats['needs_human'] += 1

        # Patch selected_row with the inline-verified (best) dialogue
        selected_row['discussion_log'] = _inl_dialogue
        selected_tactic_scale          = _inl_matrix

    # ── Finalise selected_row ──────────────────────────────────────────────────
    selected_row['matrix_tactic_scale'] = selected_tactic_scale
    selected_row['LLM_used']            = llm_used
    selected_row['Claude_Reasoning']    = result['reasoning']
    verified_results.append(selected_row)

    # Queue for Step 4b retry only when Layer 1 produced a silent failure
    if result.get('needs_retry'):
        retry_rows.append({'idx': idx, 'game_id': game_id, 'round_id': round_id, 'reason': result['correction_mode']})
        print(f"  ⚠️  Queued for Step 4b retry (reason: {result['correction_mode']})")

    # Create detailed criteria record for CSV
    criteria_record = {
        'ID': score_id,
        'Gemini31_Coherence':   format_check(result['gemini3_criteria']['coherence']),
        'Gemini31_History':     format_check(result['gemini3_criteria']['history']),
        'Gemini31_Matrix':      format_check(result['gemini3_criteria']['matrix']),
        'Gemini31_Authenticity': format_check(result['gemini3_criteria']['authenticity']),
        'Gemini31_Format':      format_check(result['gemini3_criteria']['format']),
        'Gemini31_Total':       f"{result['gemini3_total']} of 5",
        'GPT52_Coherence':      format_check(result['gpt52_criteria']['coherence']),
        'GPT52_History':        format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix':         format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity':   format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format':         format_check(result['gpt52_criteria']['format']),
        'GPT52_Total':          f"{result['gpt52_total']} of 5",
        'Selected_LLM':         llm_used,
        'Correction_Mode':      result['correction_mode'] or 'None',
        'Claude_Reasoning':     result['reasoning'],
        'Gemini31_Criteria_Explanations': json.dumps(result['gemini3_criteria_explanations']),
        'GPT52_Criteria_Explanations':    json.dumps(result['gpt52_criteria_explanations']),
        'Inline_Recheck_Mode':         _inl_mode,
        'Inline_Recheck_Score':        _inl_score,
        'Inline_Recheck_Reasoning':    _inl_reason,
        'Inline_Recheck_Explanations': _inl_expls,
        'Claude_Response': raw_response  # full raw response for debugging
    }
    criteria_results.append(criteria_record)

    # Print summary
    print(f"  ✓ Gemini-3.1: {result['gemini3_total']}/5 pass ({', '.join([k for k,v in result['gemini3_criteria'].items() if v])})")
    print(f"  ✓ GPT-5.2:    {result['gpt52_total']}/5 pass ({', '.join([k for k,v in result['gpt52_criteria'].items() if v])})")
    print(f"  ✓ Selected: {llm_used}")
    if result['correction_mode']:
        print(f"  ✓ Correction Mode: {result['correction_mode']}")
    print(f"  ✓ Reason: {result['reasoning'][:100]}...")

    # Rate limit to avoid API throttling
    time.sleep(1)

print("\n" + "=" * 80)
print(f"Verification Summary ({min(TEST_LIMIT, len(gemini3_df))} games):")
print(f"  Total verified:                          {verification_stats['total_games']}")
print(f"  Gemini-3.1 selected (original):          {verification_stats['response_1_selected']}")
print(f"  GPT-5.2 selected (original):             {verification_stats['response_2_selected']}")
print(f"  Pairwise tiebreaker used:                {verification_stats['pairwise_tiebreaker']}")
print(f"  Gemini-3.1 + Targeted correction:        {verification_stats['targeted_correction_1']}")
print(f"  GPT-5.2 + Targeted correction:           {verification_stats['targeted_correction_2']}")
print(f"  Claude 4.5 full custom generation:       {verification_stats['full_custom_generation']}")
print(f"  Flagged NEEDS_HUMAN (inline L2 failed):  {verification_stats['needs_human']}")
print(f"  Errors (API/parse failure):              {verification_stats['errors']}")
print(f"  Queued for Step 4b retry:                {len(retry_rows)}")


Starting verification of 250 games...

[1/250] Verifying game G001/2...
  ✓ Gemini-3.1: 3/5 pass (history, authenticity, format)
  ✓ GPT-5.2:    5/5 pass (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2
  ✓ Reason: Response B passes all 5 criteria (5/5) while Response A fails coherence and matrix alignment (3/5). ...

[2/250] Verifying game G002/2...
  ✓ Gemini-3.1: 4/5 pass (coherence, history, authenticity, format)
  ✓ GPT-5.2:    5/5 pass (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2
  ✓ Reason: Response B passes all 5 criteria (5/5) while Response A fails the matrix criterion (4/5) due to P5's...

[3/250] Verifying game G003/2...
  ✓ Gemini-3.1: 3/5 pass (history, matrix, format)
  ✓ GPT-5.2:    5/5 pass (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2
  ✓ Reason: Response B passes all 5 criteria (5/5) while Response A fails on coherence and authenticity (3/5). R...

[4/250] Verifying game G004/2...
  ✓ Gemi

## Step 4b: Retry Failed Rows

Retries rows collected in `retry_rows` during Step 4a. Three failure reasons are tracked:

| Reason | Cause | Entry in `verified_results`? |
|---|---|---|
| `api_error` | `verify_dialogue_pair` returned `None` (API / parse failure) | ❌ No — must append if retry succeeds |
| `Targeted` | `tc_dialogue` was `None`; targeted correction silently fell back to uncorrected base dialogue | ✅ Yes — patch in-place |
| `Full_Custom` | `cd_dialogue` was empty; `selected_dialogue` stored as `None` | ✅ Yes — patch in-place |

After a successful retry, any `Targeted` or `Full_Custom` correction is immediately passed through the same inline Layer 2 recheck (up to 3 attempts) used in Step 4a. Rows that still fail Layer 2 after all 3 attempts are flagged `NEEDS_HUMAN` and appended to `verification_stats['needs_human']`.

Rows that still fail Layer 1 itself after retry receive a hard fallback (raw model dialogue by higher score), logged with a `-Fallback` suffix in `LLM_used`.

In [20]:

# ─── Step 4b: Retry failed rows ──────────────────────────────────────────────
#
# Uses retry_rows list built during Step 4a (cell 12).
# Failure types:
#   'api_error'   — no entry in verified_results; append if retry succeeds
#   'Targeted'    — entry exists but dialogue is uncorrected fallback; patch
#   'Full_Custom' — entry exists but discussion_log is None; patch
#
# Inline Layer 2 recheck runs for every successful Targeted / Full_Custom retry,
# identical to the 3-attempt logic used in Step 4a.
# ─────────────────────────────────────────────────────────────────────────────

_CRITERIA_ORDER = ['coherence', 'history', 'matrix', 'authenticity', 'format']
_LABEL_MAP = {
    'RESPONSE_A':        'Gemini-3.1',
    'CORRECTION_A':      'Gemini-3.1-Claude-4.5',
    'RESPONSE_B':        'GPT-5.2',
    'CORRECTION_B':      'GPT-5.2-Claude-4.5',
    'CUSTOM_GENERATION': 'Claude-4.5',
}

def _make_criteria_record(score_id, result, llm_used, correction_mode, raw_response,
                          inl_mode='N/A', inl_score='N/A', inl_reason='N/A', inl_expls='N/A'):
    """Build a criteria_results record from a verify_dialogue_pair result dict.
    Pass inl_* arguments when an inline Layer 2 recheck was performed."""
    return {
        'ID':                             score_id,
        'Gemini31_Coherence':             format_check(result['gemini3_criteria']['coherence']),
        'Gemini31_History':               format_check(result['gemini3_criteria']['history']),
        'Gemini31_Matrix':                format_check(result['gemini3_criteria']['matrix']),
        'Gemini31_Authenticity':          format_check(result['gemini3_criteria']['authenticity']),
        'Gemini31_Format':                format_check(result['gemini3_criteria']['format']),
        'Gemini31_Total':                 f"{result['gemini3_total']} of 5",
        'GPT52_Coherence':                format_check(result['gpt52_criteria']['coherence']),
        'GPT52_History':                  format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix':                   format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity':             format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format':                   format_check(result['gpt52_criteria']['format']),
        'GPT52_Total':                    f"{result['gpt52_total']} of 5",
        'Selected_LLM':                   llm_used,
        'Correction_Mode':                correction_mode,
        'Claude_Reasoning':               result['reasoning'],
        'Gemini31_Criteria_Explanations': json.dumps(result['gemini3_criteria_explanations']),
        'GPT52_Criteria_Explanations':    json.dumps(result['gpt52_criteria_explanations']),
        'Inline_Recheck_Mode':            inl_mode,
        'Inline_Recheck_Score':           inl_score,
        'Inline_Recheck_Reasoning':       inl_reason,
        'Inline_Recheck_Explanations':    inl_expls,
        'Claude_Response':                raw_response,
    }

def _make_error_criteria_stub(score_id, raw_response):
    """Build an all-zero criteria stub for a row where the API failed entirely."""
    zero_expl = json.dumps({c: '' for c in _CRITERIA_ORDER})
    return {
        'ID':                             score_id,
        'Gemini31_Coherence':             0,
        'Gemini31_History':               0,
        'Gemini31_Matrix':                0,
        'Gemini31_Authenticity':          0,
        'Gemini31_Format':                0,
        'Gemini31_Total':                 '0 of 5',
        'GPT52_Coherence':                0,
        'GPT52_History':                  0,
        'GPT52_Matrix':                   0,
        'GPT52_Authenticity':             0,
        'GPT52_Format':                   0,
        'GPT52_Total':                    '0 of 5',
        'Selected_LLM':                   'Error',
        'Correction_Mode':                'Error',
        'Claude_Reasoning':               f'(api_error: {raw_response})',
        'Gemini31_Criteria_Explanations': zero_expl,
        'GPT52_Criteria_Explanations':    zero_expl,
        'Inline_Recheck_Mode':            'N/A',
        'Inline_Recheck_Score':           'N/A',
        'Inline_Recheck_Reasoning':       'N/A',
        'Inline_Recheck_Explanations':    'N/A',
        'Claude_Response':                '',
    }

if not retry_rows:
    print("✅ No rows to retry — all verifications succeeded.")
else:
    n_api    = sum(1 for r in retry_rows if r['reason'] == 'api_error')
    n_tc     = sum(1 for r in retry_rows if r['reason'] == 'Targeted')
    n_cd     = sum(1 for r in retry_rows if r['reason'] == 'Full_Custom')
    print(f"⚠️  Retrying {len(retry_rows)} row(s): "
          f"{n_api} api_error, {n_tc} Targeted, {n_cd} Full_Custom\n")
    for entry in retry_rows:
        print(f"  • {entry['game_id']}/{entry['round_id']} — {entry['reason']}")
    print()

    succeeded    = 0
    still_failed = 0

    for entry in sorted(retry_rows, key=lambda e: e['idx']):
        idx      = entry['idx']
        game_id  = entry['game_id']
        round_id = entry['round_id']
        reason   = entry['reason']
        score_id = f"{game_id}/{round_id}"

        print(f" Retrying {score_id} (reason: {reason})...")

        player_roles       = gemini3_df.iloc[idx]['player_roles']
        protagonist        = gemini3_df.iloc[idx]['role_id']
        public_history     = gemini3_df.iloc[idx]['public_history']
        prior_summary_gold = prior_summary_lookup.get(gemini3_df.iloc[idx]['game_id'], '')
        dialogue_gemini3   = gemini3_df.iloc[idx]['discussion_log']
        tactic_scale_gem3  = gemini3_df.iloc[idx]['matrix_tactic_scale']
        dialogue_gpt52     = gpt52_df.iloc[idx]['discussion_log']
        tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']

        result, raw_response = verify_dialogue_pair(
            game_id, round_id,
            player_roles, protagonist, public_history,
            dialogue_gemini3, tactic_scale_gem3,
            dialogue_gpt52,   tactic_scale_gpt52,
            prior_summary_gold=prior_summary_gold,
        )

        # Find existing position in verified_results / criteria_results
        existing_pos = next(
            (i for i, r in enumerate(verified_results) if r.get('game_id') == game_id),
            None
        )
        existing_cr_pos = next(
            (i for i, cr in enumerate(criteria_results) if cr['ID'] == score_id),
            None
        )

        # ── API / parse error on retry ─────────────────────────────────────
        if result is None:
            print(f"  ✗ Retry failed: {raw_response}")
            if existing_pos is not None:
                print(f"  ↩  Keeping previous fallback entry in verified_results (pos {existing_pos}).")
            else:
                print(f"  ✗  No entry created for {game_id} — game will be absent from CSV.")

            stub = _make_error_criteria_stub(score_id, raw_response)
            if existing_cr_pos is not None:
                criteria_results[existing_cr_pos] = stub
            else:
                criteria_results.append(stub)
            print(f"  ↩  Criteria stub (all-zero) written for {score_id}.")

            still_failed += 1
            print()
            time.sleep(1)
            continue

        # ── Still a silent failure after retry → hard fallback ─────────────
        if result.get('needs_retry') or not result.get('selected_dialogue'):
            g3_total = result['gemini3_total']
            gp_total = result['gpt52_total']
            if g3_total >= gp_total:
                fallback_dialogue = dialogue_gemini3
                fallback_ts       = tactic_scale_gem3
                fallback_llm      = 'Gemini-3.1-Fallback'
                fallback_base     = gemini3_df.iloc[idx].copy()
            else:
                fallback_dialogue = dialogue_gpt52
                fallback_ts       = tactic_scale_gpt52
                fallback_llm      = 'GPT-5.2-Fallback'
                fallback_base     = gpt52_df.iloc[idx].copy()
            fallback_base['discussion_log']      = fallback_dialogue
            fallback_base['matrix_tactic_scale'] = fallback_ts
            fallback_base['LLM_used']             = fallback_llm
            fallback_base['Claude_Reasoning']     = '(fallback: still empty after retry)'

            if existing_pos is not None:
                verified_results[existing_pos] = fallback_base
                print(f"  ↩  Patched verified_results [pos {existing_pos}] with fallback: {fallback_llm}")
            else:
                verified_results.append(fallback_base)
                print(f"  ↩  Appended fallback entry: {fallback_llm}")

            # Write actual scores to criteria_results (we have them even if dialogue failed)
            cr = _make_criteria_record(score_id, result, fallback_llm, 'Fallback', raw_response)
            if existing_cr_pos is not None:
                criteria_results[existing_cr_pos] = cr
            else:
                criteria_results.append(cr)
            print(f"  ↩  Criteria record (real scores) written with Correction_Mode=Fallback.")

            still_failed += 1
            print()
            time.sleep(1)
            continue

        # ── Successful retry: build selected_row ──────────────────────────
        rec      = result['recommendation']
        llm_used = _LABEL_MAP.get(rec, rec)

        if rec == 'RESPONSE_A':
            selected_row = gemini3_df.iloc[idx].copy()
        elif rec == 'CORRECTION_A':
            selected_row = gemini3_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']
        elif rec == 'RESPONSE_B':
            selected_row = gpt52_df.iloc[idx].copy()
        elif rec == 'CORRECTION_B':
            selected_row = gpt52_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']
        else:  # CUSTOM_GENERATION
            selected_row = gpt52_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']

        selected_row['matrix_tactic_scale'] = (
            result['selected_tactic_scale'] or format_matrix_tactic_scale(tactic_scale_gpt52)
        )

        g3_pass = ', '.join(k for k in _CRITERIA_ORDER if result['gemini3_criteria'].get(k))
        gp_pass = ', '.join(k for k in _CRITERIA_ORDER if result['gpt52_criteria'].get(k))
        print(f"  ✓ Gemini-3.1: {result['gemini3_total']}/5 ({g3_pass})")
        print(f"  ✓ GPT-5.2:    {result['gpt52_total']}/5 ({gp_pass})")
        print(f"  ✓ Selected: {llm_used} | Mode: {result['correction_mode'] or 'None'}")

        # ── Inline Layer 2 recheck (Targeted / Full_Custom corrections) ────
        _inl_mode   = 'N/A'
        _inl_score  = 'N/A'
        _inl_reason = 'N/A'
        _inl_expls  = 'N/A'

        if (
            result.get('correction_mode') in ('Targeted', 'Full_Custom')
            and result.get('selected_dialogue')
        ):
            _inl_dialogue = result['selected_dialogue']
            _inl_matrix   = selected_row['matrix_tactic_scale']

            for _att in range(1, 4):
                time.sleep(1)
                print(f"  🔍 Inline L2 recheck attempt {_att}/3...")
                _vsd, _ = verify_single_dialogue(
                    game_id, round_id, player_roles, protagonist, public_history,
                    _inl_dialogue, _inl_matrix,
                    prior_summary_gold=prior_summary_gold,
                )

                if _vsd is None:
                    print(f"  ⚠️  Inline recheck API error (attempt {_att})")
                    if _att == 3:
                        print(f"  ⚠️  All 3 inline attempts failed — flagging NEEDS_HUMAN")
                        llm_used   += '-NEEDS_HUMAN'
                        _inl_mode   = 'NEEDS_HUMAN'
                        _inl_score  = '0 of 5'
                        _inl_reason = 'API error on all 3 inline attempts'
                        _inl_expls  = '{}'
                        verification_stats['needs_human'] += 1
                    continue

                _inl_score  = f"{_vsd['total']} of 5"
                _inl_reason = _vsd['reasoning']
                _inl_expls  = json.dumps(_vsd['criteria_explanations'])

                if _vsd['total'] == 5:
                    print(f"  ✓ Inline L2 recheck: 5/5 — accepted (attempt {_att})")
                    _inl_dialogue = _vsd['selected_dialogue']
                    _inl_matrix   = _vsd['selected_tactic_scale']
                    _inl_mode     = 'INLINE_ACCEPTED'
                    break
                elif _att < 3:
                    print(f"  ⚠️  Inline L2: {_vsd['total']}/5 — fix applied, retrying (attempt {_att})")
                    _inl_dialogue = _vsd['selected_dialogue']
                    _inl_matrix   = _vsd['selected_tactic_scale']
                    _inl_mode     = 'INLINE_FIXED'
                else:  # _att == 3 and total < 5
                    print(f"  ⚠️  Inline L2: {_vsd['total']}/5 after 3 attempts — flagging NEEDS_HUMAN")
                    _inl_dialogue = _vsd['selected_dialogue']
                    _inl_matrix   = _vsd['selected_tactic_scale']
                    llm_used   += '-NEEDS_HUMAN'
                    _inl_mode   = 'NEEDS_HUMAN'
                    verification_stats['needs_human'] += 1

            # Patch selected_row with the inline-verified (best) dialogue
            selected_row['discussion_log']      = _inl_dialogue
            selected_row['matrix_tactic_scale'] = _inl_matrix

        # ── Finalise selected_row fields (after llm_used may have changed) ─
        selected_row['LLM_used']         = llm_used
        selected_row['Claude_Reasoning'] = result['reasoning']

        # Patch or append verified_results
        if existing_pos is not None:
            old_llm = verified_results[existing_pos].get('LLM_used', '?')
            verified_results[existing_pos] = selected_row
            print(f"  ✓ Patched verified_results [pos {existing_pos}] → {llm_used} (was: {old_llm})")
        else:
            verified_results.insert(idx, selected_row)
            print(f"  ✓ Inserted new entry at pos {idx} → {llm_used}")

        # Patch or insert criteria_results (pass inline recheck vars)
        new_cr = _make_criteria_record(
            score_id, result, llm_used, result['correction_mode'] or 'None', raw_response,
            inl_mode=_inl_mode, inl_score=_inl_score, inl_reason=_inl_reason, inl_expls=_inl_expls,
        )
        if existing_cr_pos is not None:
            criteria_results[existing_cr_pos] = new_cr
        else:
            criteria_results.insert(idx, new_cr)

        succeeded += 1
        print()
        time.sleep(1)

    print("─" * 60)
    print(f"✅ Retry complete: {succeeded} succeeded, {still_failed} still failed.")
    if still_failed:
        print("   Persistent failures use raw-model fallback dialogue (hard fallback) or zero-stub (api_error).")
    print(f"   verified_results: {len(verified_results)} rows | criteria_results: {len(criteria_results)} rows")
    print("   verified_results and criteria_results are ready to save.")

✅ No rows to retry — all verifications succeeded.


## Step 4c: NEEDS_HUMAN Summary — Inline Layer 2 Recheck

In [21]:
# ─── Step 4c: Layer 2 is now inline in Step 4a ───────────────────────────────
#
# verify_single_dialogue() is called INLINE within the Step 4a main loop,
# immediately after each Targeted correction or Full_Custom generation.
# Up to 3 inline attempts per row. After 3 failures → NEEDS_HUMAN flag.
#
# This cell now serves as a post-run diagnostic: shows which rows were
# flagged NEEDS_HUMAN and summarises inline recheck outcomes.
# ─────────────────────────────────────────────────────────────────────────────

if not criteria_results:
    print("⚠️  No criteria_results yet — run Step 4a first.")
else:
    needs_human_rows = [
        cr for cr in criteria_results
        if cr.get('Inline_Recheck_Mode') == 'NEEDS_HUMAN'
    ]
    inline_accepted  = sum(1 for cr in criteria_results if cr.get('Inline_Recheck_Mode') == 'INLINE_ACCEPTED')
    inline_fixed     = sum(1 for cr in criteria_results if cr.get('Inline_Recheck_Mode') == 'INLINE_FIXED')
    inline_na        = sum(1 for cr in criteria_results if cr.get('Inline_Recheck_Mode') == 'N/A')

    print("=" * 60)
    print("Step 4c — Inline Layer 2 Recheck Summary")
    print("=" * 60)
    print(f"  N/A (direct selections, no recheck):    {inline_na}")
    print(f"  INLINE_ACCEPTED (passed 5/5 on recheck): {inline_accepted}")
    print(f"  INLINE_FIXED (passed after >=1 fix):     {inline_fixed}")
    print(f"  NEEDS_HUMAN (all 3 attempts failed):     {len(needs_human_rows)}")
    print()

    if needs_human_rows:
        print("Rows requiring human review:")
        for cr in needs_human_rows:
            print(f"  {cr['ID']:12s}  LLM: {cr['Selected_LLM']:35s}  Score: {cr['Inline_Recheck_Score']}")
        print()
        print("Action: review these dialogues manually in verified_r2_seeds_combined.csv")
        print("        (they appear with '-NEEDS_HUMAN' suffix in LLM_used column)")
    else:
        print("✅ All corrected/custom rows passed inline Layer 2 recheck — no human review needed.")


Step 4c — Inline Layer 2 Recheck Summary
  N/A (direct selections, no recheck):    243
  INLINE_ACCEPTED (passed 5/5 on recheck): 7
  INLINE_FIXED (passed after >=1 fix):     0
  NEEDS_HUMAN (all 3 attempts failed):     0

✅ All corrected/custom rows passed inline Layer 2 recheck — no human review needed.


## Step 5: Create Output DataFrames

Convert verification results into structured DataFrames.

In [22]:
# Create Output DataFrames
# Create DataFrame from verified results (full data)
if verified_results:
    verified_df = pd.DataFrame(verified_results)
    
    # Ensure LLM_used is last column
    cols = verified_df.columns.tolist()
    cols = [col for col in cols if col != 'LLM_used'] + ['LLM_used']
    verified_df = verified_df[cols]
    
    print(f"\n✓ Verified DataFrame (combined) shape: {verified_df.shape}")
    print(f"✓ Columns: {list(verified_df.columns)}")
    print(f"\nFirst verified row (LLM_used column):")
    print(verified_df[['game_id', 'round_id', 'LLM_used']].head())
else:
    print("No results to create verified output CSV")
    verified_df = None

# Create DataFrame from criteria results (detailed per-criterion tracking)
if criteria_results:
    criteria_df = pd.DataFrame(criteria_results)
    
    print(f"\n✓ Criteria DataFrame shape: {criteria_df.shape}")
    print(f"✓ Columns: {list(criteria_df.columns)}")
    print(f"\nFirst criteria records:")
    print(criteria_df.head())
    
    # Calculate aggregate statistics
    print(f"\n{'='*60}")
    print("AGGREGATE STATISTICS (for paper analysis):")
    print(f"{'='*60}")
    
    # Helper function to count passes
    def count_passes(df, prefix):
        coherence = (df[f'{prefix}_Coherence'] == 1).sum()
        history = (df[f'{prefix}_History'] == 1).sum()
        matrix = (df[f'{prefix}_Matrix'] == 1).sum()
        authenticity = (df[f'{prefix}_Authenticity'] == 1).sum()
        format_check = (df[f'{prefix}_Format'] == 1).sum()
        total = len(df)
        return {
            'Coherence': f"{coherence}/{total} ({coherence/total*100:.1f}%)",
            'History': f"{history}/{total} ({history/total*100:.1f}%)",
            'Matrix': f"{matrix}/{total} ({matrix/total*100:.1f}%)",
            'Authenticity': f"{authenticity}/{total} ({authenticity/total*100:.1f}%)",
            'Format': f"{format_check}/{total} ({format_check/total*100:.1f}%)"
        }
    
    gemini31_stats = count_passes(criteria_df, 'Gemini31')
    gpt52_stats = count_passes(criteria_df, 'GPT52')
    
    print("\nGemini-3.1 Pass Rates:")
    for criterion, stat in gemini31_stats.items():
        print(f"  {criterion:15s}: {stat}")
    
    print("\nGPT-5.2 Pass Rates:")
    for criterion, stat in gpt52_stats.items():
        print(f"  {criterion:15s}: {stat}")
else:
    print("No results to create criteria output CSV")
    criteria_df = None


✓ Verified DataFrame (combined) shape: (250, 11)
✓ Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'Claude_Reasoning', 'LLM_used']

First verified row (LLM_used column):
  game_id  round_id LLM_used
0    G001         2  GPT-5.2
1    G002         2  GPT-5.2
2    G003         2  GPT-5.2
3    G004         2  GPT-5.2
4    G005         2  GPT-5.2

✓ Criteria DataFrame shape: (250, 23)
✓ Columns: ['ID', 'Gemini31_Coherence', 'Gemini31_History', 'Gemini31_Matrix', 'Gemini31_Authenticity', 'Gemini31_Format', 'Gemini31_Total', 'GPT52_Coherence', 'GPT52_History', 'GPT52_Matrix', 'GPT52_Authenticity', 'GPT52_Format', 'GPT52_Total', 'Selected_LLM', 'Correction_Mode', 'Claude_Reasoning', 'Gemini31_Criteria_Explanations', 'GPT52_Criteria_Explanations', 'Inline_Recheck_Mode', 'Inline_Recheck_Score', 'Inline_Recheck_Reasoning', 'Inline_Recheck_Explanations', 'Claude_Response']

First criteria

## Step 6: Output Files

Files saved by the verification process:

1. **Datasets/verified/verified_r2_seeds_combined.csv**: Full dialogue rows with columns:
   - All original columns from input CSVs
   - `LLM_used`: Source indicator (Gemini-3.1, GPT-5.2, Gemini-3.1-Claude-4.5, GPT-5.2-Claude-4.5, or Claude-4.5)
   - `Claude_Reasoning`: Claude's explanation for the selection decision

2. **Datasets/verified/verified_r2_criteria_scores.csv**: Detailed per-criterion validation tracking with columns:
   - `ID`: Game/Round identifier
   - `Gemini31_Coherence`, `Gemini31_History`, `Gemini31_Matrix`, `Gemini31_Authenticity`, `Gemini31_Format`: Binary checks (1 = pass, 0 = fail)
   - `Gemini31_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `GPT52_Coherence`, `GPT52_History`, `GPT52_Matrix`, `GPT52_Authenticity`, `GPT52_Format`: Binary checks (1 = pass, 0 = fail)
   - `GPT52_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `Selected_LLM`: Which model's output was used
   - `Correction_Mode`: None, Pairwise_Tiebreaker, Targeted, or Full_Custom
   - `Inline_Recheck_Mode`: N/A (no correction), INLINE_ACCEPTED, INLINE_FIXED, or NEEDS_HUMAN
   - `Inline_Recheck_Score`: score from Layer 2 recheck (e.g. "5 of 5"), or N/A
   - `Inline_Recheck_Reasoning`: Claude's reasoning from the recheck, or N/A
   - `Inline_Recheck_Explanations`: JSON per-criterion explanations from recheck, or N/A
   - `Claude_Reasoning`: Claude's explanation for the decision
   - `Gemini31_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)
   - `GPT52_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)

**For Paper Analysis:** The criteria CSV enables reporting like: "Gemini-3.1 achieved 95% format compliance (214/225 games)" and identifying which criteria are most challenging for each model. Criteria explanations provide insights into why each criterion passed or failed.

In [23]:
# Save Verified Results
if verified_df is not None and criteria_df is not None:
    # Save combined verified results
    verified_df.to_csv(OUTPUT_FILE_COMBINED, index=False)
    print(f"✓ Verified results saved to {OUTPUT_FILE_COMBINED}")
    print(f"  Total rows: {len(verified_df)}")
    
    # Save criteria tracking with detailed per-criterion scores
    criteria_df.to_csv(OUTPUT_FILE_CRITERIA, index=False)
    print(f"\n✓ Criteria tracking saved to {OUTPUT_FILE_CRITERIA}")
    print(f"  Total records: {len(criteria_df)}")
    
    # Display samples
    print(f"\nSample of verified results (LLM_used distribution):")
    print(verified_df['LLM_used'].value_counts())
    
    print(f"\nSample of criteria records (first 5):")
    print(criteria_df.head(5))
    
    # Display correction mode distribution
    print(f"\nCorrection mode distribution:")
    print(criteria_df['Correction_Mode'].value_counts())
else:
    print("No verified results to save")

✓ Verified results saved to Datasets/verified/verified_r2_seeds_combined.csv
  Total rows: 250

✓ Criteria tracking saved to Datasets/verified/verified_r2_criteria_scores.csv
  Total records: 250

Sample of verified results (LLM_used distribution):
LLM_used
GPT-5.2                  234
Gemini-3.1                 9
GPT-5.2-Claude-4.5         5
Gemini-3.1-Claude-4.5      1
Claude-4.5                 1
Name: count, dtype: int64

Sample of criteria records (first 5):
       ID  Gemini31_Coherence  Gemini31_History  Gemini31_Matrix  \
0  G001/2                   0                 1                0   
1  G002/2                   1                 1                0   
2  G003/2                   0                 1                1   
3  G004/2                   1                 1                0   
4  G005/2                   1                 1                0   

   Gemini31_Authenticity  Gemini31_Format Gemini31_Total  GPT52_Coherence  \
0                      1                1     

## Next Steps: Full Verification

After testing this notebook with the first 2 games and reviewing the results:

1. **Review Sample Results**: Check that binary validation logic works correctly
2. **Check Criteria Distribution**: Examine which criteria pass/fail most frequently
3. **Validate Corrections**: 
   - Check "Targeted" corrections properly fix only the failing criterion
   - Verify "Full_Custom" generations when both models score ≤3/5
   - Confirm "Pairwise_Tiebreaker" selections when both score 5/5
4. **Verify Decision Logic**: Ensure the system correctly handles all scenarios:
   - Both 5/5 → Pairwise comparison
   - One 5/5, other <5 → Selects 5/5
   - At least one 4/5 → Targeted correction (pairwise judgment if both 4/5, same as 5/5 tiebreaker)
   - Both ≤3/5 → Full custom generation
5. **Scale Up**: Change `TEST_LIMIT = 2` to `TEST_LIMIT = len(gemini3_df)` to process all 225 games
6. **Analyze Results**: Use criteria CSV to calculate per-criterion pass rates for your paper
7. **Final Merge**: Combine verified results with 25 manually generated seeds for complete 250-seed dataset

**Important Notes:**
- Current test uses TEST_LIMIT = 2 to validate the system
- Full verification of 225 games will take ~4-5 minutes (1 sec delay + longer max_tokens)
- Per-criterion tracking enables detailed analysis for paper (e.g., "Format: 98% pass rate")

## Comprehensive Analysis: Binary Multi-Criteria Validation Results

Detailed analysis of the verified r1 criteria scores including:
- Per-criterion passing rates for both LLMs
- Average performance metrics
- LLM selection statistics
- Correction mode distribution
- Most/least reliable criteria
- Additional insights

In [26]:

# ── COMPREHENSIVE ANALYSIS (All Sections Combined) ───────────────────────────
import pandas as pd
import numpy as np

criteria_df = pd.read_csv('Datasets/verified/verified_r2_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']

# Pre-build score lists used by all later sections
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

print("="*80)
print("COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results")
print("="*80)
print(f"\nTotal Records Analyzed: {len(criteria_df)}")

# ── SECTION 1: PER-CRITERION PASSING RATES ───────────────────────────────────
print("\n" + "="*80)
print("1. PER-CRITERION PASSING RATES")
print("="*80)

print("\nGemini-3.1 Passing Rates:")
gemini31_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'Gemini31_{c}'].sum() / len(criteria_df)) * 100
    gemini31_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'Gemini31_{c}'].sum())}/{len(criteria_df)})")

print("\nGPT-5.2 Passing Rates:")
gpt52_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'GPT52_{c}'].sum() / len(criteria_df)) * 100
    gpt52_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'GPT52_{c}'].sum())}/{len(criteria_df)})")

print("\nComparative Analysis (GPT-5.2 vs Gemini-3.1):")
for c in criteria_names:
    diff = gpt52_passing_rates[c] - gemini31_passing_rates[c]
    winner = "GPT-5.2" if diff > 0 else ("Gemini-3.1" if diff < 0 else "TIE")
    print(f"  {c:15s}: {diff:+6.2f}% difference (Winner: {winner})")

# ── SECTION 2: AVERAGE PERFORMANCE METRICS ───────────────────────────────────
gemini31_avg = np.mean(gemini31_scores)
gpt52_avg    = np.mean(gpt52_scores)
gemini31_std = np.std(gemini31_scores)
gpt52_std    = np.std(gpt52_scores)
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("2. AVERAGE PERFORMANCE METRICS")
print("="*80)

print(f"\nGemini-3.1:")
print(f"  Average Score: {gemini31_avg:.3f}/5 (SD: {gemini31_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gemini31_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gemini31_scores))*100:5.2f}%)")

print(f"\nGPT-5.2:")
print(f"  Average Score: {gpt52_avg:.3f}/5 (SD: {gpt52_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gpt52_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gpt52_scores))*100:5.2f}%)")

print(f"\nOverall Comparison:")
if gpt52_avg > gemini31_avg:
    print(f"  GPT-5.2 outperforms Gemini-3.1 by {gpt52_avg - gemini31_avg:.3f} points ({((gpt52_avg - gemini31_avg) / gemini31_avg) * 100:.2f}%)")
elif gemini31_avg > gpt52_avg:
    print(f"  Gemini-3.1 outperforms GPT-5.2 by {gemini31_avg - gpt52_avg:.3f} points")
else:
    print(f"  Both models perform equally: {gemini31_avg:.3f}/5")

# ── SECTION 3: LLM SELECTION STATISTICS ──────────────────────────────────────
selection_counts = criteria_df['Selected_LLM'].value_counts()

print("\n" + "="*80)
print("3. LLM SELECTION STATISTICS")
print("="*80)
print("\nSelection Breakdown:")
for sel, cnt in selection_counts.items():
    print(f"  {sel:30s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

# ── SECTION 4: CORRECTION MODE DISTRIBUTION ──────────────────────────────────
correction_counts = criteria_df['Correction_Mode'].value_counts()
targeted_count    = len(criteria_df[criteria_df['Correction_Mode'] == 'Targeted'])
both_4of5         = sum((g4 == 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gemini31_4of5= sum((g4 == 4 and g5 != 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gpt52_4of5   = sum((g4 != 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("4. CORRECTION MODE DISTRIBUTION")
print("="*80)
print("\nCorrection Mode Breakdown:")
for mode, cnt in correction_counts.items():
    print(f"  {str(mode):20s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

print(f"\nTargeted Correction Breakdown:")
print(f"  Total targeted corrections:                  {targeted_count}")
print(f"  Both scored 4/5 (pairwise judgment applied): {both_4of5:4d} ({(both_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only Gemini-3.1 scored 4/5:                  {only_gemini31_4of5:4d} ({(only_gemini31_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only GPT-5.2 scored 4/5:                     {only_gpt52_4of5:4d} ({(only_gpt52_4of5/len(criteria_df))*100:5.2f}%)")
print(f"\n  Full custom generations: {len(criteria_df[criteria_df['Correction_Mode'] == 'Full_Custom']):4d}")
print(f"  Targeted corrections:    {targeted_count:4d}")

# ── SECTION 5: CRITERIA RANKING ──────────────────────────────────────────────
gemini31_sorted = sorted(gemini31_passing_rates.items(), key=lambda x: x[1])
gpt52_sorted    = sorted(gpt52_passing_rates.items(),    key=lambda x: x[1])

print("\n" + "="*80)
print("5. MOST FAILING AND PASSING CRITERIA")
print("="*80)
print("\nGemini-3.1:")
print(f"  Most Challenging: {gemini31_sorted[0][0]} ({gemini31_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gemini31_sorted[-1][0]} ({gemini31_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gemini31_sorted])}")
print("\nGPT-5.2:")
print(f"  Most Challenging: {gpt52_sorted[0][0]} ({gpt52_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gpt52_sorted[-1][0]} ({gpt52_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gpt52_sorted])}")

# ── SECTION 5a: DETAILED SELECTION BREAKDOWN ─────────────────────────────────
# Use str.startswith to capture -NEEDS_HUMAN suffix variants
gemini31_direct    = (criteria_df['Selected_LLM'] == 'Gemini-3.1').sum()
gemini31_corrected = criteria_df['Selected_LLM'].str.startswith('Gemini-3.1-Claude-4.5').sum()
gpt52_direct       = (criteria_df['Selected_LLM'] == 'GPT-5.2').sum()
gpt52_corrected    = criteria_df['Selected_LLM'].str.startswith('GPT-5.2-Claude-4.5').sum()
claude_full_custom = criteria_df['Selected_LLM'].str.startswith('Claude-4.5').sum()
needs_human_count  = criteria_df['Selected_LLM'].str.endswith('-NEEDS_HUMAN').sum()

print("\n" + "="*80)
print("5a. DETAILED SELECTION BREAKDOWN")
print("="*80)
print(f"\n  Gemini-3.1 selected directly:          {gemini31_direct:4d} ({(gemini31_direct/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 corrected by Claude:        {gemini31_corrected:4d} ({(gemini31_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 total usage:                {gemini31_direct + gemini31_corrected:4d} ({((gemini31_direct + gemini31_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  GPT-5.2 selected directly:             {gpt52_direct:4d} ({(gpt52_direct/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 corrected by Claude:           {gpt52_corrected:4d} ({(gpt52_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 total usage:                   {gpt52_direct + gpt52_corrected:4d} ({((gpt52_direct + gpt52_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  Claude full custom (both <=3/5):       {claude_full_custom:4d} ({(claude_full_custom/len(criteria_df))*100:5.2f}%)")
print(f"  Pairwise tiebreaker (both 5/5):        {both_perfect:4d} ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"  \u26a0\ufe0f  Flagged NEEDS_HUMAN:                  {needs_human_count:4d} ({(needs_human_count/len(criteria_df))*100:5.2f}%)")
print(f"  Total accounted for:                   {gemini31_direct + gemini31_corrected + gpt52_direct + gpt52_corrected + claude_full_custom:4d} (verify = {len(criteria_df)})")

# ── SECTION 6: ADDITIONAL INSIGHTS ───────────────────────────────────────────
one_perfect = sum((g4 == 5) != (g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
both_low    = sum((g4 <= 3 and g5 <= 3)   for g4, g5 in zip(gemini31_scores, gpt52_scores))
agreement   = sum(g4 == g5                 for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("6. ADDITIONAL INSIGHTS")
print("="*80)
print(f"\nBoth models scored 5/5:       {both_perfect:4d} games ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Exactly one model scored 5/5: {one_perfect:4d} games ({(one_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Both models scored <=3/5:     {both_low:4d} games ({(both_low/len(criteria_df))*100:5.2f}%)")
print(f"Score Agreement (same total): {agreement:4d} games ({(agreement/len(criteria_df))*100:5.2f}%)")

print("\nCriteria where both models fail together:")
for c in criteria_names:
    both_fail = int(((criteria_df[f'Gemini31_{c}'] == 0) & (criteria_df[f'GPT52_{c}'] == 0)).sum())
    print(f"  {c:15s}: {both_fail:4d} games ({(both_fail/len(criteria_df))*100:5.2f}%)")

print("\nPer-Criterion Agreement (both pass or both fail):")
for c in criteria_names:
    agree_count = int((criteria_df[f'Gemini31_{c}'] == criteria_df[f'GPT52_{c}']).sum())
    print(f"  {c:15s}: {agree_count:4d} agreements ({(agree_count/len(criteria_df))*100:5.2f}%)")

# ── SECTION 7: INLINE LAYER 2 RECHECK SUMMARY ──────────────────────────────── 
print("\n" + "="*80)
print("7. INLINE LAYER 2 RECHECK SUMMARY")
print("="*80)
inline_mode_counts = criteria_df['Inline_Recheck_Mode'].value_counts(dropna=True)
needs_human_inline = (criteria_df['Inline_Recheck_Mode'] == 'NEEDS_HUMAN').sum()
na_count           = criteria_df['Inline_Recheck_Mode'].isna().sum() + (criteria_df['Inline_Recheck_Mode'] == 'N/A').sum()
print(f"\n  Rows with no inline recheck needed (N/A):  {na_count:4d} ({(na_count/len(criteria_df))*100:5.2f}%)")
print(f"  Inline recheck triggered:                  {len(criteria_df) - na_count:4d} ({((len(criteria_df) - na_count)/len(criteria_df))*100:5.2f}%)")
print(f"\n  Inline Recheck Mode Breakdown:")
for _mode, _cnt in inline_mode_counts.items():
    print(f"    {str(_mode):25s}: {_cnt:4d} ({(_cnt/len(criteria_df))*100:5.2f}%)")
print(f"\n  \u26a0\ufe0f  NEEDS_HUMAN (require manual review): {needs_human_inline:4d} ({(needs_human_inline/len(criteria_df))*100:5.2f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results

Total Records Analyzed: 250

1. PER-CRITERION PASSING RATES

Gemini-3.1 Passing Rates:
  Coherence      :  80.80% (202/250)
  History        :  99.20% (248/250)
  Matrix         :  90.00% (225/250)
  Authenticity   :  83.20% (208/250)
  Format         : 100.00% (250/250)

GPT-5.2 Passing Rates:
  Coherence      :  99.60% (249/250)
  History        :  99.60% (249/250)
  Matrix         :  97.20% (243/250)
  Authenticity   : 100.00% (250/250)
  Format         : 100.00% (250/250)

Comparative Analysis (GPT-5.2 vs Gemini-3.1):
  Coherence      : +18.80% difference (Winner: GPT-5.2)
  History        :  +0.40% difference (Winner: GPT-5.2)
  Matrix         :  +7.20% difference (Winner: GPT-5.2)
  Authenticity   : +16.80% difference (Winner: GPT-5.2)
  Format         :  +0.00% difference (Winner: TIE)

2. AVERAGE PERFORMANCE METRICS

Gemini-3.1:
  Average Score: 4.532/5 (SD: 0.722)
  Score Distribution:
    0/5:    0 games ( 0.00

In [27]:
# ── SETUP: Load data and shared variables ────────────────────────────────────
import pandas as pd
import numpy as np

criteria_df = pd.read_csv('Datasets/verified/verified_r2_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']

# Pre-build score lists used by all later sections
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

print("="*80)
print("COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results")
print("="*80)
print(f"\nTotal Records Analyzed: {len(criteria_df)}")

# ── SECTION 1: PER-CRITERION PASSING RATES ───────────────────────────────────
print("\n" + "="*80)
print("1. PER-CRITERION PASSING RATES")
print("="*80)

print("\nGemini-3.1 Passing Rates:")
gemini31_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'Gemini31_{c}'].sum() / len(criteria_df)) * 100
    gemini31_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'Gemini31_{c}'].sum())}/{len(criteria_df)})")

print("\nGPT-5.2 Passing Rates:")
gpt52_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'GPT52_{c}'].sum() / len(criteria_df)) * 100
    gpt52_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'GPT52_{c}'].sum())}/{len(criteria_df)})")

print("\nComparative Analysis (GPT-5.2 vs Gemini-3.1):")
for c in criteria_names:
    diff = gpt52_passing_rates[c] - gemini31_passing_rates[c]
    winner = "GPT-5.2" if diff > 0 else ("Gemini-3.1" if diff < 0 else "TIE")
    print(f"  {c:15s}: {diff:+6.2f}% difference (Winner: {winner})")

COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results

Total Records Analyzed: 250

1. PER-CRITERION PASSING RATES

Gemini-3.1 Passing Rates:
  Coherence      :  80.80% (202/250)
  History        :  99.20% (248/250)
  Matrix         :  90.00% (225/250)
  Authenticity   :  83.20% (208/250)
  Format         : 100.00% (250/250)

GPT-5.2 Passing Rates:
  Coherence      :  99.60% (249/250)
  History        :  99.60% (249/250)
  Matrix         :  97.20% (243/250)
  Authenticity   : 100.00% (250/250)
  Format         : 100.00% (250/250)

Comparative Analysis (GPT-5.2 vs Gemini-3.1):
  Coherence      : +18.80% difference (Winner: GPT-5.2)
  History        :  +0.40% difference (Winner: GPT-5.2)
  Matrix         :  +7.20% difference (Winner: GPT-5.2)
  Authenticity   : +16.80% difference (Winner: GPT-5.2)
  Format         :  +0.00% difference (Winner: TIE)


In [28]:
# ── SECTION 2: AVERAGE PERFORMANCE METRICS ───────────────────────────────────
gemini31_avg = np.mean(gemini31_scores)
gpt52_avg    = np.mean(gpt52_scores)
gemini31_std = np.std(gemini31_scores)
gpt52_std    = np.std(gpt52_scores)
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*80)
print("2. AVERAGE PERFORMANCE METRICS")
print("="*80)

print(f"\nGemini-3.1:")
print(f"  Average Score: {gemini31_avg:.3f}/5 (SD: {gemini31_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gemini31_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gemini31_scores))*100:5.2f}%)")

print(f"\nGPT-5.2:")
print(f"  Average Score: {gpt52_avg:.3f}/5 (SD: {gpt52_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gpt52_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gpt52_scores))*100:5.2f}%)")

print(f"\nOverall Comparison:")
if gpt52_avg > gemini31_avg:
    print(f"  GPT-5.2 outperforms Gemini-3.1 by {gpt52_avg - gemini31_avg:.3f} points ({((gpt52_avg - gemini31_avg) / gemini31_avg) * 100:.2f}%)")
elif gemini31_avg > gpt52_avg:
    print(f"  Gemini-3.1 outperforms GPT-5.2 by {gemini31_avg - gpt52_avg:.3f} points")
else:
    print(f"  Both models perform equally: {gemini31_avg:.3f}/5")

2. AVERAGE PERFORMANCE METRICS

Gemini-3.1:
  Average Score: 4.532/5 (SD: 0.722)
  Score Distribution:
    0/5:    0 games ( 0.00%)
    1/5:    0 games ( 0.00%)
    2/5:    1 games ( 0.40%)
    3/5:   31 games (12.40%)
    4/5:   52 games (20.80%)
    5/5:  166 games (66.40%)

GPT-5.2:
  Average Score: 4.964/5 (SD: 0.207)
  Score Distribution:
    0/5:    0 games ( 0.00%)
    1/5:    0 games ( 0.00%)
    2/5:    0 games ( 0.00%)
    3/5:    1 games ( 0.40%)
    4/5:    7 games ( 2.80%)
    5/5:  242 games (96.80%)

Overall Comparison:
  GPT-5.2 outperforms Gemini-3.1 by 0.432 points (9.53%)


In [29]:
# ── SECTION 3: LLM SELECTION STATISTICS ──────────────────────────────────────
selection_counts = criteria_df['Selected_LLM'].value_counts()

print("="*80)
print("3. LLM SELECTION STATISTICS")
print("="*80)
print("\nSelection Breakdown:")
for sel, cnt in selection_counts.items():
    print(f"  {sel:30s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

3. LLM SELECTION STATISTICS

Selection Breakdown:
  GPT-5.2                       :  234 (93.60%)
  Gemini-3.1                    :    9 ( 3.60%)
  GPT-5.2-Claude-4.5            :    5 ( 2.00%)
  Gemini-3.1-Claude-4.5         :    1 ( 0.40%)
  Claude-4.5                    :    1 ( 0.40%)


In [30]:

# ── SECTION 4: CORRECTION MODE DISTRIBUTION ──────────────────────────────────
import pandas as pd
import numpy as np

criteria_df = pd.read_csv('Datasets/verified/verified_r2_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

correction_counts = criteria_df['Correction_Mode'].value_counts()
targeted_count    = len(criteria_df[criteria_df['Correction_Mode'] == 'Targeted'])
both_4of5         = sum((g4 == 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gemini31_4of5= sum((g4 == 4 and g5 != 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gpt52_4of5   = sum((g4 != 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*60)
print("4. CORRECTION MODE DISTRIBUTION")
print("="*60)
print("\nCorrection Mode Breakdown:")
for mode, count in correction_counts.items():
    print(f"  {str(mode):20s}: {count:4d} ({(count/len(criteria_df))*100:5.2f}%)")

print(f"\nTargeted Correction Breakdown:")
print(f"  Total targeted corrections:                  {targeted_count}")
print(f"  Both scored 4/5 (pairwise judgment applied): {both_4of5:4d} ({(both_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only Gemini-3.1 scored 4/5:                  {only_gemini31_4of5:4d} ({(only_gemini31_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only GPT-5.2 scored 4/5:                     {only_gpt52_4of5:4d} ({(only_gpt52_4of5/len(criteria_df))*100:5.2f}%)")
needs_human_count = (criteria_df['Inline_Recheck_Mode'] == 'NEEDS_HUMAN').sum()
print(f"\n  Full custom generations: {len(criteria_df[criteria_df['Correction_Mode'] == 'Full_Custom']):4d}")
print(f"  Targeted corrections:    {targeted_count:4d}")
print(f"  NEEDS_HUMAN (inline L2 failed): {needs_human_count:4d} ({(needs_human_count/len(criteria_df))*100:5.2f}%)")
print(f"  Note: NEEDS_HUMAN rows are still counted under their Correction_Mode (Targeted/Full_Custom)")

4. CORRECTION MODE DISTRIBUTION

Correction Mode Breakdown:
  Pairwise_Tiebreaker :  165 (66.00%)
  Targeted            :    6 ( 2.40%)
  Full_Custom         :    1 ( 0.40%)

Targeted Correction Breakdown:
  Total targeted corrections:                  6
  Both scored 4/5 (pairwise judgment applied):    6 ( 2.40%)
  Only Gemini-3.1 scored 4/5:                    46 (18.40%)
  Only GPT-5.2 scored 4/5:                        1 ( 0.40%)

  Full custom generations:    1
  Targeted corrections:       6
  NEEDS_HUMAN (inline L2 failed):    0 ( 0.00%)
  Note: NEEDS_HUMAN rows are still counted under their Correction_Mode (Targeted/Full_Custom)


In [31]:
# ── SECTION 5: CRITERIA RANKING + SECTION 5a: DETAILED SELECTION ─────────────
# Reload shared state in case this cell is run after a kernel restart or
# after Section 4 (which re-loads criteria_df but doesn't rebuild passing_rates).
import pandas as pd
import numpy as np

criteria_df = pd.read_csv('Datasets/verified/verified_r2_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']

gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

gemini31_passing_rates = {
    c: (criteria_df[f'Gemini31_{c}'].sum() / len(criteria_df)) * 100
    for c in criteria_names
}
gpt52_passing_rates = {
    c: (criteria_df[f'GPT52_{c}'].sum() / len(criteria_df)) * 100
    for c in criteria_names
}
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

gemini31_sorted = sorted(gemini31_passing_rates.items(), key=lambda x: x[1])
gpt52_sorted    = sorted(gpt52_passing_rates.items(),    key=lambda x: x[1])

print("="*80)
print("5. MOST FAILING AND PASSING CRITERIA")
print("="*80)
print("\nGemini-3.1:")
print(f"  Most Challenging: {gemini31_sorted[0][0]} ({gemini31_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gemini31_sorted[-1][0]} ({gemini31_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gemini31_sorted])}")
print("\nGPT-5.2:")
print(f"  Most Challenging: {gpt52_sorted[0][0]} ({gpt52_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gpt52_sorted[-1][0]} ({gpt52_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gpt52_sorted])}")

# Use str.startswith to capture -NEEDS_HUMAN suffix variants
gemini31_direct    = (criteria_df['Selected_LLM'] == 'Gemini-3.1').sum()
gemini31_corrected = criteria_df['Selected_LLM'].str.startswith('Gemini-3.1-Claude-4.5').sum()
gpt52_direct       = (criteria_df['Selected_LLM'] == 'GPT-5.2').sum()
gpt52_corrected    = criteria_df['Selected_LLM'].str.startswith('GPT-5.2-Claude-4.5').sum()
claude_full_custom = criteria_df['Selected_LLM'].str.startswith('Claude-4.5').sum()
needs_human_count  = criteria_df['Selected_LLM'].str.endswith('-NEEDS_HUMAN').sum()

print("\n" + "="*80)
print("5a. DETAILED SELECTION BREAKDOWN")
print("="*80)
print(f"\n  Gemini-3.1 selected directly:          {gemini31_direct:4d} ({(gemini31_direct/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 corrected by Claude:        {gemini31_corrected:4d} ({(gemini31_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 total usage:                {gemini31_direct + gemini31_corrected:4d} ({((gemini31_direct + gemini31_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  GPT-5.2 selected directly:             {gpt52_direct:4d} ({(gpt52_direct/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 corrected by Claude:           {gpt52_corrected:4d} ({(gpt52_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 total usage:                   {gpt52_direct + gpt52_corrected:4d} ({((gpt52_direct + gpt52_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  Claude full custom (both <=3/5):       {claude_full_custom:4d} ({(claude_full_custom/len(criteria_df))*100:5.2f}%)")
print(f"  Pairwise tiebreaker (both 5/5):        {both_perfect:4d} ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"  Flagged NEEDS_HUMAN:                  {needs_human_count:4d} ({(needs_human_count/len(criteria_df))*100:5.2f}%)")
print(f"  Total accounted for:                   {gemini31_direct + gemini31_corrected + gpt52_direct + gpt52_corrected + claude_full_custom:4d} (verify = {len(criteria_df)})")

5. MOST FAILING AND PASSING CRITERIA

Gemini-3.1:
  Most Challenging: Coherence (80.80% pass rate)
  Most Reliable:    Format (100.00% pass rate)
  Full Ranking (worst→best): Coherence (80.8%), Authenticity (83.2%), Matrix (90.0%), History (99.2%), Format (100.0%)

GPT-5.2:
  Most Challenging: Matrix (97.20% pass rate)
  Most Reliable:    Format (100.00% pass rate)
  Full Ranking (worst→best): Matrix (97.2%), Coherence (99.6%), History (99.6%), Authenticity (100.0%), Format (100.0%)

5a. DETAILED SELECTION BREAKDOWN

  Gemini-3.1 selected directly:             9 ( 3.60%)
  Gemini-3.1 corrected by Claude:           1 ( 0.40%)
  Gemini-3.1 total usage:                  10 ( 4.00%)

  GPT-5.2 selected directly:              234 (93.60%)
  GPT-5.2 corrected by Claude:              5 ( 2.00%)
  GPT-5.2 total usage:                    239 (95.60%)

  Claude full custom (both <=3/5):          1 ( 0.40%)
  Pairwise tiebreaker (both 5/5):         165 (66.00%)
  Flagged NEEDS_HUMAN:             

In [32]:
# ── SECTION 6: ADDITIONAL INSIGHTS ─────────────────────────────────────────────────────
# Self-contained reload — safe to run after kernel restart or out-of-order.
import pandas as pd
import numpy as np

criteria_df     = pd.read_csv('Datasets/verified/verified_r2_criteria_scores.csv')
criteria_names  = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]
both_perfect    = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

one_perfect = sum((g4 == 5) != (g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
both_low    = sum((g4 <= 3 and g5 <= 3)   for g4, g5 in zip(gemini31_scores, gpt52_scores))
agreement   = sum(g4 == g5                 for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*80)
print("6. ADDITIONAL INSIGHTS")
print("="*80)
print(f"\nBoth models scored 5/5:       {both_perfect:4d} games ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Exactly one model scored 5/5: {one_perfect:4d} games ({(one_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Both models scored <=3/5:     {both_low:4d} games ({(both_low/len(criteria_df))*100:5.2f}%)")
print(f"Score Agreement (same total): {agreement:4d} games ({(agreement/len(criteria_df))*100:5.2f}%)")

print("\nCriteria where both models fail together:")
for c in criteria_names:
    both_fail = int(((criteria_df[f'Gemini31_{c}'] == 0) & (criteria_df[f'GPT52_{c}'] == 0)).sum())
    print(f"  {c:15s}: {both_fail:4d} games ({(both_fail/len(criteria_df))*100:5.2f}%)")

print("\nPer-Criterion Agreement (both pass or both fail):")
for c in criteria_names:
    agree_count = int((criteria_df[f'Gemini31_{c}'] == criteria_df[f'GPT52_{c}']).sum())
    print(f"  {c:15s}: {agree_count:4d} agreements ({(agree_count/len(criteria_df))*100:5.2f}%)")

# ── SECTION 7: INLINE LAYER 2 RECHECK SUMMARY ──────────────────────────────── 
print("\n" + "="*80)
print("7. INLINE LAYER 2 RECHECK SUMMARY")
print("="*80)
inline_mode_counts = criteria_df['Inline_Recheck_Mode'].value_counts(dropna=True)
needs_human_inline = (criteria_df['Inline_Recheck_Mode'] == 'NEEDS_HUMAN').sum()
na_count           = criteria_df['Inline_Recheck_Mode'].isna().sum() + (criteria_df['Inline_Recheck_Mode'] == 'N/A').sum()
print(f"\n  Rows with no inline recheck needed (N/A):  {na_count:4d} ({(na_count/len(criteria_df))*100:5.2f}%)")
print(f"  Inline recheck triggered:                  {len(criteria_df) - na_count:4d} ({((len(criteria_df) - na_count)/len(criteria_df))*100:5.2f}%)")
print(f"\n  Inline Recheck Mode Breakdown:")
for _mode, _cnt in inline_mode_counts.items():
    print(f"    {str(_mode):25s}: {_cnt:4d} ({(_cnt/len(criteria_df))*100:5.2f}%)")
print(f"\n   NEEDS_HUMAN (require manual review): {needs_human_inline:4d} ({(needs_human_inline/len(criteria_df))*100:5.2f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

6. ADDITIONAL INSIGHTS

Both models scored 5/5:        165 games (66.00%)
Exactly one model scored 5/5:   78 games (31.20%)
Both models scored <=3/5:        1 games ( 0.40%)
Score Agreement (same total):  171 games (68.40%)

Criteria where both models fail together:
  Coherence      :    1 games ( 0.40%)
  History        :    0 games ( 0.00%)
  Matrix         :    7 games ( 2.80%)
  Authenticity   :    0 games ( 0.00%)
  Format         :    0 games ( 0.00%)

Per-Criterion Agreement (both pass or both fail):
  Coherence      :  203 agreements (81.20%)
  History        :  247 agreements (98.80%)
  Matrix         :  232 agreements (92.80%)
  Authenticity   :  208 agreements (83.20%)
  Format         :  250 agreements (100.00%)

7. INLINE LAYER 2 RECHECK SUMMARY

  Rows with no inline recheck needed (N/A):   243 (97.20%)
  Inline recheck triggered:                     7 ( 2.80%)

  Inline Recheck Mode Breakdown:
    INLINE_ACCEPTED          :    7 ( 2.80%)

   NEEDS_HUMAN (require manual r